# Reproducing Duszkiewicz et al. (2024) from dandiset 000939

Paper. Duszkiewicz, Orhan, Skromne Carrasco, Brown, Owczarek, Vite, Wood and Peyrache
(2024), "Local origin of excitatory-inhibitory tuning equivalence in a cortical network",
Nature Neuroscience 27, 782-792, doi:10.1038/s41593-024-01588-5. The claims file
(`claims/000939/10.1038-s41593-024-01588-5.yaml`) was written from the version of record
(PMC11001581) with its Methods, Extended Data legends and Supplementary Information, and from
the paper's MATLAB code on figshare (doi:10.6084/m9.figshare.24921252, version 3).

Dataset. DANDI:000939, version 0.261016.2310 (doi:10.48324/dandi.000939/0.261016.2310),
31 NWB files, one PoSub session per mouse, streamed from the archive. The paper cites no
version of the dandiset (its data statement names the figshare item), so the most recent
published version is pinned. Its 31 files hold the same sessions as version 0.250207.0025,
with the raw voltage added in 2026; the data check below shows that the unit, spike and
head-direction sample counts of every file are the same in both versions.

Outside data. Claims marked `data: elsewhere` in the claims file use the figshare item
above, version 3: Dataset_5 (ADN recordings), Dataset_6 (optogenetic sessions with light over
ADN), Dataset_4 (Neuropixel session) and Dataset_3 (cue rotation), each streamed and read in
memory.

| Claim | Tested here |
|---|---|
| cell-classes-counts | tested |
| ed2b-excitatory-hd-information | tested |
| fig2a-fs-hd-information | tested |
| fig2b-decoding-from-fs | tested |
| fig2c-geometry-independent | tested |
| ed2hi-within-epoch-stability | tested |
| fig2f-cue-rotation | tested, figshare Dataset_3 |
| fig3c-adn-broader | tested, figshare Dataset_5 |
| fig3-ten-components-power | tested |
| fig3e-fourier-signatures-differ | tested, figshare Dataset_5 |
| fig3f-posub-hd-closer-to-fs | tested, figshare Dataset_5 |
| ed3k-robust-to-controls | tested, figshare Dataset_5 |
| fig3g-first-three-components | tested |
| fig3h-dominant-component | tested |
| ed3g-fs-spectra-heterogeneous | tested (ADN part reported) |
| ed3l-fs-phases-independent | tested |
| ed4a-autocorrelogram-symmetries | tested |
| ed4cd-isomap-posub-central | tested, figshare Dataset_5 |
| fig4-random-connectivity-simulation | blocked: simulation code and parameters not available |
| fig4f-synaptic-weights-dispersed | tested |
| fig4h-model-fit | blocked: fitting code not available, input numbers and initial values not given |
| ed7-adn-disinhibition | tested, figshare Dataset_6 |
| fig5b-posub-rates-increase | tested |
| ed7gj-tuning-preserved | tested |
| fig5e-hd-multiplicative-fs-additive | tested |
| ed8cd-multiplicative-detectable | tested |
| ed8e-adn-mixed-gain | tested, figshare Dataset_6 |
| fig6c-coupling-preserved-wake-rem | tested |
| ed9d-coupling-follows-tuning | tested |
| fig7b-wake-isomap-tuning | tested |
| ed10b-ring-topology | tested |
| fig7d-rem-isomap-tuning | tested |
| fig1ac-posub-border | tested, figshare Dataset_4 |
| ed6-trn-innervates-adn | blocked: anatomical images not deposited |

The notebook streams about 2.7 GB of processed data from the archive and 4 GB of tarballs
from figshare; it takes about half an hour, most of it in the GLM fits.

## Data check

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import posub

T0 = time.time()
print("dandiset", posub.DANDISET, "version", posub.VERSION)
S = posub.load_all()
print(f"streamed {len(S)} files in {time.time() - T0:.0f} s")
rows = []
for s in S:
    sq = s["epochs"]["wake_square"][0]
    rem = (s["sleep"].get("rem", np.zeros((0, 2)))[:, 1] - s["sleep"].get("rem", np.zeros((0, 2)))[:, 0]).sum()
    rows.append(dict(subject=s["subject"], session=s["session_id"], orientation=s["orientation"],
                     units=len(s["spikes"]), epochs=",".join(s["epochs"]),
                     square_s=round(sq[1] - sq[0]), rem_s=round(float(rem)),
                     pulses=len(s.get("pulses", [])), virus=s["virus"][:40], genotype=s["genotype"][12:30]))
meta = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 20, "display.max_rows", 100)
print(meta.to_string())
print("units by orientation:", meta.groupby("orientation").units.agg(["count", "sum"]).to_dict())
# units and ranges of the variables used
hd_all = np.concatenate([s["hd"] for s in S]); pos_all = np.concatenate([s["pos"] for s in S])
print("head direction unit:", {s["hd_unit"] for s in S}, "range", np.nanmin(hd_all).round(3), np.nanmax(hd_all).round(3),
      "NaN fraction", np.isnan(hd_all).mean().round(4))
print("position unit:", {s["pos_unit"] for s in S}, "range x", np.nanmin(pos_all[:, 0]).round(1), np.nanmax(pos_all[:, 0]).round(1),
      "y", np.nanmin(pos_all[:, 1]).round(1), np.nanmax(pos_all[:, 1]).round(1))
dts = np.array([np.median(np.diff(s["hd_t"])) for s in S])
print("HD sampling interval (s), min and max over files:", dts.min().round(5), dts.max().round(5))
print("spike time resolution attribute:", {s["spike_res"] for s in S})
ttp = np.concatenate([s["ttp"] for s in S])
print("trough-to-peak (ms): n", len(ttp), "NaN", np.isnan(ttp).sum(), "range", np.nanmin(ttp), np.nanmax(ttp))
# do spikes and tracking share a clock: spikes fall inside the recording, tracking inside its epochs
for s in S:
    last = max(sp.max() for sp in s["spikes"] if len(sp)); end = max(b for v in s["epochs"].values() for a, b in v)
    assert last <= end + 1, s["subject"]
print("all spike times fall within the session's epochs")
# HD sampled only during exploration epochs?
for s in S:
    fin = np.isfinite(s["hd"]); t = s["hd_t"][fin]
    wake = [iv for k, v in s["epochs"].items() if k.startswith("wake") for iv in v]
    s["_hd_out"] = (~posub.in_intervals(t, wake)).mean()
print("fraction of tracked HD samples outside wake epochs, max over files:", max(s["_hd_out"] for s in S).round(4))
# duplicate check: identical spike trains across files
sig = {}
for s in S:
    for i, sp in enumerate(s["spikes"]):
        key = (len(sp), round(float(sp[0]), 5) if len(sp) else None, round(float(sp[-1]), 5) if len(sp) else None)
        sig.setdefault(key, []).append((s["subject"], i))
dups = {k: v for k, v in sig.items() if len(v) > 1}
print("units with identical spike count, first and last spike in two places:", len(dups))
hd_sig = {}
for s in S:
    hd_sig.setdefault((len(s["hd"]), round(float(np.nansum(s["hd"])), 3)), []).append(s["subject"])
print("files with identical HD series:", [v for v in hd_sig.values() if len(v) > 1])
# the previous published version with the same sessions (0.250207.0025, processed data only):
# units and spike counts per file, against the pinned version
import h5py, remfile
from concurrent.futures import ThreadPoolExecutor
def _units(a):
    f = h5py.File(remfile.File(posub.s3_url(a["asset_id"])), "r")
    return a["path"].split("/")[0], len(f["units/id"]), f["units/spike_times"].shape[0], f["processing/behavior/CompassDirection/head-direction/data"].shape[0]
with ThreadPoolExecutor(8) as ex:
    old = {k: (n, ns, nh) for k, n, ns, nh in ex.map(_units, posub.assets("0.250207.0025"))}
new = {s["path"].split("/")[0]: (len(s["spikes"]), sum(len(x) for x in s["spikes"]), len(s["hd"])) for s in S}
print("subjects in 0.250207.0025 and the pinned version:", len(old), len(new), "; same subjects:", set(old) == set(new))
print("files whose unit count, spike count or HD sample count differ between the versions:", [k for k in new if old.get(k) != new[k]])

### Tuning curves of every unit

Computed once for the analyses below: square-arena curves (1 degree bins, 3 degree smoothing,
speed above 2 cm/s), their time-reversed controls, the two halves of the epoch, the
triangle-arena curves, the 2 degree smoothing variant (code), and curves at 10 cm/s
(Extended Data Fig. 3k). Rates are spike counts over the whole square epoch (code,
GetMeanRates: Rate(S, wake1Ep)).

In [ ]:
U = []  # one row per unit
TC = {}  # per session: dict of arrays (360, n units)
for si, s in enumerate(S):
    t, ang = s["hd_t"], s["hd"]
    a, b = s["epochs"]["wake_square"][0]
    n = len(s["spikes"])
    rate = np.array([((sp >= a) & (sp < b)).sum() / (b - a) for sp in s["spikes"]])
    m = posub.good_samples(s, "wake_square")
    d = {}
    for sd in (3, 2):
        d[f"sq{sd}"], occ = posub.tuning_curves(s["spikes"], t, ang, m, sd=sd)
        d[f"sq{sd}_rev"], occ_r = posub.tuning_curves(s["spikes"], t, ang, m, sd=sd, reverse=True)
        m1, m2 = posub.split_halves(t, m)
        d[f"h1_{sd}"], _ = posub.tuning_curves(s["spikes"], t, ang, m1, sd=sd)
        d[f"h2_{sd}"], _ = posub.tuning_curves(s["spikes"], t, ang, m2, sd=sd)
        d[f"h1_{sd}_rev"], _ = posub.tuning_curves(s["spikes"], t, ang, m1, sd=sd, reverse=True)
        d[f"h2_{sd}_rev"], _ = posub.tuning_curves(s["spikes"], t, ang, m2, sd=sd, reverse=True)
        if sd == 3:
            info = posub.hd_info(d["sq3"], occ)
            info_rev = posub.hd_info(d["sq3_rev"], occ_r)
            move_s = m.sum() * np.median(np.diff(t))
    m10 = posub.good_samples(s, "wake_square", speed_th=10.0)
    d["sq3_v10"], _ = posub.tuning_curves(s["spikes"], t, ang, m10)
    rate_tri = np.full(n, np.nan)
    if "wake_triangle" in s["epochs"]:
        ta, tb = s["epochs"]["wake_triangle"][0]
        rate_tri = np.array([((sp >= ta) & (sp < tb)).sum() / (tb - ta) for sp in s["spikes"]])
        mt = posub.good_samples(s, "wake_triangle")
        for sd in (3, 2):
            d[f"tri{sd}"], _ = posub.tuning_curves(s["spikes"], t, ang, mt, sd=sd)
            d[f"tri{sd}_rev"], _ = posub.tuning_curves(s["spikes"], t, ang, mt, sd=sd, reverse=True)
    TC[si] = d
    for i in range(n):
        U.append(dict(sess=si, subject=s["subject"], unit=i, orientation=s["orientation"], rate=rate[i],
                      rate_tri=rate_tri[i], ttp=s["ttp"][i], info=info[i], info_rev=info_rev[i],
                      lab_ex=s["lab_ex"][i], lab_fs=s["lab_fs"][i], lab_hd=s["lab_hd"][i], move_s=move_s))
U = pd.DataFrame(U)
U["gd"] = (U.rate > posub.RATE_MIN) & np.isfinite(U.ttp)
print(f"{len(U)} units; {U.gd.sum()} viable (rate > 0.5 Hz in the square epoch and a trough-to-peak, i.e. a negative waveform)")
print("time moving above 2 cm/s in the square epoch (s), min and max over sessions:",
      U.groupby("sess").move_s.first().min().round(0), U.groupby("sess").move_s.first().max().round(0))
print(f"tuning curves done at {time.time() - T0:.0f} s")

## cell-classes-counts

> PoSub units split into 1,835 putative excitatory cells and 427 putative FS cells by firing
> rate and waveform, and 1,602 excitatory cells (87%) were HD cells with HD information above
> 0.2 bits per spike; the vertical recordings gave 931 units from 14 mice and the parallel ones
> 1,999 units from 18 mice.

Classes as the claims file fixes them (Methods): viable units (rate at least 0.5 Hz in the
square epoch, negative waveform, read as a finite trough-to-peak), FS below 0.4 ms and above
10 Hz, excitatory above 0.4 ms and below 10 Hz; the code's 0.35 ms cut (with its >= 10 Hz and
<= 0.35 ms) is a variant. HD cells are excitatory cells above 0.2 bits per spike, and, as a
second reading, above the 99th percentile of the excitatory cells' time-reversed HD information.

In [ ]:
classify = posub.classify
U["ex"], U["fs"] = classify(U)
U["hd"] = U.ex & (U["info"] > posub.HD_INFO_TH)
p99 = np.percentile(U["info_rev"][U.ex], 99)
rows = []
for name, cut, code in [("Methods, 0.4 ms", 0.4, False), ("code, 0.35 ms", 0.35, True)]:
    ex, fs = classify(U, cut, code)
    for thname, th in [("0.2 bits/spike", 0.2), (f"99th pct of reversed ({p99:.3f})", p99)]:
        hd = ex & (U["info"] > th)
        rows.append(dict(classes=name, hd_threshold=thname, excitatory=int(ex.sum()), FS=int(fs.sum()),
                         HD=int(hd.sum()), HD_fraction=round(hd.sum() / ex.sum(), 3)))
print(pd.DataFrame(rows).to_string())
print("units by orientation:", U.groupby("orientation").size().to_dict(),
      "sessions:", U.groupby("orientation").sess.nunique().to_dict())
print("dandiset's own labels: excitatory", int(U.lab_ex.sum()), "FS", int(U.lab_fs.sum()), "HD", int(U.lab_hd.sum()))
print("agreement with the dandiset labels (main reading): ex", (U.ex == U.lab_ex).mean().round(4),
      "fs", (U.fs == U.lab_fs).mean().round(4), "hd", (U.hd == U.lab_hd).mean().round(4))
exc, fsc = classify(U, 0.35, True)
print("agreement with the dandiset labels (code reading): ex", (exc == U.lab_ex).mean().round(4),
      "fs", (fsc == U.lab_fs).mean().round(4), "hd", ((exc & (U["info"] > 0.2)) == U.lab_hd).mean().round(4))
print("units neither excitatory nor FS among viable ones (main reading):", int((U.gd & ~U.ex & ~U.fs).sum()))
print(U.groupby("subject")[["ex", "fs", "hd"]].sum().T.to_string())
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
g = U[U.gd]
ax[0].scatter(g.ttp, g.rate, s=3, c=np.where(g.fs, "tab:orange", np.where(g.ex, "tab:purple", "grey")))
ax[0].set_yscale("log"); ax[0].set_xlabel("trough-to-peak (ms)"); ax[0].set_ylabel("rate in square (Hz)")
ax[1].hist(np.log10(U["info"][U.gd].clip(1e-4)), 80); ax[1].set_xlabel("log10 HD information (bits/spike), viable units")
plt.tight_layout(); plt.show()
print("plotted ranges: ttp", g.ttp.min(), g.ttp.max(), "rate", g.rate.min().round(2), g.rate.max().round(1),
      "; units with information below 1e-4 drawn at -4:", int((U["info"][U.gd] < 1e-4).sum()))

**Result: partially reproduced.** Under the Methods' classes there are 1,661 excitatory cells (expected 1,835 +- 184), 430 FS cells (427 +- 43), 1,512 HD cells (1,602 +- 160) and an HD fraction of 0.910 (0.87 or 0.85 +- 0.087), all within tolerance, as are the code's classes (1,699, 415, 1,532, 0.902) and the HD threshold read as the 99th percentile of the time-reversed control (0.189 bits per spike). The vertical recordings hold 931 units from 14 mice, as the paper gives. The parallel recordings hold 1,760 units from 17 mice, below 1,999 - 200: the dandiset has 17 of the paper's 18 parallel sessions. The dandiset's own labels agree with the code's classes for 99.9% of units.

### Statistics used below
`signrank` (helper) is the two-sided Wilcoxon signed-rank test with its normal-approximation z (MATLAB's
signrank reports this z); `ranksum` (helper) is the two-sided Mann-Whitney U test with its z. All tests
are two-tailed (Methods).

In [ ]:
signrank, ranksum, show, classify = posub.signrank, posub.ranksum, posub.show, posub.classify

## ed2b-excitatory-hd-information

> Putative excitatory PoSub cells carry more HD information than their time-reversed
> controls, and HD information across all PoSub cells is bimodal.

Log10 HD information of each excitatory cell against its time-reversed control,
Wilcoxon signed-rank (criterion; code Figure_BasicProperties).

In [ ]:
lg = lambda v: np.log10(np.clip(v, 1e-6, None))
for name, (exm, _) in {"Methods classes": (U.ex, U.fs), "code classes": classify(U, 0.35, True)}.items():
    show(signrank(lg(U["info"][exm]), lg(U["info_rev"][exm])), f"excitatory, {name}:")
print("paper: n = 1835, z = 37.1")
print("per session, fraction of excitatory cells above their own control:",
      U[U.ex].assign(up=lambda d: d["info"] > d["info_rev"]).groupby("subject").up.mean().round(2).describe()[["min", "50%", "max"]].to_dict())
h, e = np.histogram(lg(U["info"][U.gd]), 60)
fig, ax = plt.subplots(figsize=(5, 3)); ax.stairs(h, e); ax.set_xlabel("log10 HD info (bits/spike), all viable units"); plt.show()
# bimodality shown, not judged: the two largest local maxima of the smoothed histogram
from scipy.ndimage import gaussian_filter1d as _g
hs = _g(h.astype(float), 2); pk = [i for i in range(1, len(hs) - 1) if hs[i] >= hs[i - 1] and hs[i] >= hs[i + 1]]
print("local maxima of the smoothed histogram at log10 info =", [round((e[i] + e[i + 1]) / 2, 2) for i in pk], "range of values", e[0].round(2), e[-1].round(2))

**Result: reproduced.** Excitatory cells carry more HD information than their time-reversed controls (median log10 information 0.10 against -1.53, |z| = 35.3, p = 6e-273, n = 1,661; the paper gives z = 37.1 for 1,835). The histogram of HD information over all viable units has two maxima, near log10 = -1.67 and 0.19 (shown, not judged).

## fig2a-fs-hd-information

> The HD information carried by PoSub-FS cell tuning curves is higher than that of their
> time-reversed controls.

In [ ]:
for name, (_, fsm) in {"Methods classes": (U.ex, U.fs), "code classes": classify(U, 0.35, True)}.items():
    show(signrank(lg(U["info"][fsm]), lg(U["info_rev"][fsm])), f"FS, {name}:")
print("paper: n = 427, z = 17.9")
print("median HD information of PoSub-HD cells (bits/spike):", U["info"][U.hd].median().round(3),
      "; of FS cells:", U["info"][U.fs].median().round(3), "; FS controls:", U["info_rev"][U.fs].median().round(3))
print("FS cells above their own control:", int((U["info"][U.fs] > U["info_rev"][U.fs]).sum()), "of", int(U.fs.sum()))
print(U[U.fs].assign(up=lambda d: d["info"] > d["info_rev"]).groupby("subject").up.mean().round(2).to_dict())

**Result: reproduced.** FS cells carry more HD information than their time-reversed controls (median log10 information -1.70 against -2.80, |z| = 17.96, p = 4e-72, n = 430; paper z = 17.9 for 427); 425 of 430 FS cells are above their own control. The median HD information of HD cells is 1.35 bits per spike, that of FS cells 0.02.

## ed2hi-within-epoch-stability

> Tuning curves of PoSub-FS and PoSub-HD cells computed from the two halves of the
> square-arena epoch are correlated above their time-reversed controls.

Zero-lag Pearson correlation of the two halves' curves of each cell, against that of the
time-reversed halves (criterion; code GetHdTuning_xval and Figure_Stability). Smoothing 3
degrees (Methods) and 2 degrees (code).

In [ ]:
def per_unit(key):
    """Stack a per-session curve array over all units, in the order of U."""
    return np.concatenate([TC[si][key] for si in range(len(S))], axis=1)

for sd in (3, 2):
    r = posub.zero_lag_r(per_unit(f"h1_{sd}"), per_unit(f"h2_{sd}"))
    rr = posub.zero_lag_r(per_unit(f"h1_{sd}_rev"), per_unit(f"h2_{sd}_rev"))
    U[f"r_half{sd}"], U[f"r_half{sd}_rev"] = r, rr
    for cls in ("fs", "hd"):
        show(signrank(r[U[cls]], rr[U[cls]]), f"{cls.upper()}, smoothing {sd} deg:")
print("paper: FS n = 427, z = 17.8; HD n = 1602, z = 34.7")
print("cells with an undefined correlation (a flat half):", int(np.isnan(U.r_half3[U.fs | U.hd]).sum()))
print("median per session of FS half-half r:", U[U.fs].groupby("subject").r_half3.median().round(2).describe()[["min", "50%", "max"]].round(2).to_dict())

**Result: reproduced.** Correlations between the two halves of the square epoch are above the time-reversed controls for FS cells (median r 0.83 against -0.003, |z| = 17.95, p = 5e-72, n = 430) and HD cells (0.97 against -0.016, |z| = 33.7, p = 1e-248, n = 1,512), and the same with 2 degree smoothing.

## fig2c-geometry-independent

> The tuning of PoSub-FS cells is preserved between the square and triangular arenas, as it is
> for PoSub-HD cells.

Sessions with a triangle epoch. FS cells above 10 Hz and HD cells above 0.5 Hz in both arenas
(criterion; code Figure_Stability). Each session's triangle curves are shifted back by the
circular mean of its HD cells' rotations (lag of maximum square-triangle cross-correlation;
Methods). The control shifts the time-reversed triangle curves by the same amount and
correlates them with the time-reversed square curves.

In [ ]:
tri_rows = []
for sd in (3, 2):
    rs, rc = [], []
    for si, s in enumerate(S):
        if f"tri{sd}" not in TC[si]:
            continue
        u = U[U.sess == si]
        hdm = (u.hd & (u.rate > 0.5) & (u.rate_tri > 0.5)).values
        fsm = (u.fs & (u.rate > 10) & (u.rate_tri > 10)).values
        sq, tr = TC[si][f"sq{sd}"], TC[si][f"tri{sd}"]
        sqr, trr = TC[si][f"sq{sd}_rev"], TC[si][f"tri{sd}_rev"]
        lag = posub.circ_xcorr_lag(sq[:, hdm], tr[:, hdm])
        rot = int(np.round(np.rad2deg(stats.circmean(np.deg2rad(lag))))) % 360
        tr_s, trr_s = np.roll(tr, -rot, axis=0), np.roll(trr, -rot, axis=0)
        for cls, m in (("HD", hdm), ("FS", fsm)):
            r = posub.zero_lag_r(sq[:, m], tr_s[:, m]); c = posub.zero_lag_r(sqr[:, m], trr_s[:, m])
            tri_rows += [dict(sd=sd, cls=cls, subject=s["subject"], rot=rot, r=a, r_rev=b) for a, b in zip(r, c)]
T2 = pd.DataFrame(tri_rows)
print("sessions with a triangle:", T2[T2.sd == 3].subject.nunique(), "; rotation (deg) per session:",
      T2[(T2.sd == 3)].groupby("subject").rot.first().to_dict())
for sd in (3, 2):
    for cls in ("FS", "HD"):
        d = T2[(T2.sd == sd) & (T2.cls == cls)]
        show(signrank(d.r, d.r_rev), f"{cls}, smoothing {sd} deg:")
print("paper: FS n = 264, z = 14.0; HD n = 1013, z = 27.6")
print(T2[T2.sd == 3].groupby(["cls", "subject"]).r.median().unstack(0).round(2).T.to_string())

**Result: reproduced.** In the 20 sessions with a triangle, square-triangle correlations after the session's rotation correction are above their time-reversed controls for FS cells (median r 0.80 against -0.01, |z| = 14.2, p = 1e-45, n = 273) and HD cells (0.96 against 0.00, |z| = 28.9, p = 4e-184, n = 1,118); with the code's 2 degree smoothing the same holds (|z| = 14.2 and 28.9).

## fig2b-decoding-from-fs

> The animal's HD can be decoded accurately from PoSub-FS cell spiking with a Bayesian decoder
> built on their tuning curves, and decoding error falls as more FS cells are used.

As GetBayesianDecoding.m (criterion): the square epoch from first to last tracked sample is
split into two halves of equal time; tuning curves (360 bins, 3 degree smoothing) from the
first half at speed above 2 cm/s; spike counts in 100 ms bins smoothed with a Gaussian of 2
bins s.d.; decoding over the second half, all times; error the absolute angular difference
from the HD sample nearest each bin centre, median per session. Chance: the same with the
HD time-reversed over the epoch, for the curves and for the error.

In [ ]:
from scipy.ndimage import gaussian_filter1d
dec = []
for si, s in enumerate(S):
    u = U[U.sess == si]
    t, ang = s["hd_t"], s["hd"]
    a, b = posub.tracked_epoch(s, "wake_square")
    in_ep = (t >= a) & (t <= b)
    ang_rev = ang.copy(); ang_rev[in_ep] = ang[in_ep][::-1]
    mid = (a + b) / 2
    m1 = posub.good_samples(s, "wake_square") & (t < mid)
    edges = np.arange(a, b, 0.1); cen = edges[:-1] + 0.05
    sec = cen >= mid
    j = np.clip(np.searchsorted(t, cen[sec]), 1, len(t) - 1); j = np.where(np.abs(t[j - 1] - cen[sec]) < np.abs(t[j] - cen[sec]), j - 1, j)
    row = dict(subject=s["subject"], n_fs=int(u.fs.sum()), n_hd=int(u.hd.sum()))
    for cls in ("fs", "hd"):
        idx = np.where(u[cls].values)[0]
        if len(idx) == 0:
            continue
        sp = [s["spikes"][i] for i in idx]
        Q = gaussian_filter1d(posub.bin_counts(sp, edges), 2, axis=0)
        for kind, aa in (("real", ang), ("rev", ang_rev)):
            tc, _ = posub.tuning_curves(sp, t, aa, m1 & np.isfinite(aa))
            th = np.deg2rad(posub.bayes_decode(tc, Q[sec], 0.1))
            err = np.abs(posub.angdiff(th, aa[j]))
            ok = np.isfinite(aa[j])
            row[f"{cls}_{kind}"] = np.rad2deg(np.median(err[ok]))
    dec.append(row)
dec = pd.DataFrame(dec)
print(dec.round(1).to_string())
print(f"mean over {len(dec)} sessions of median error (deg): FS {dec.fs_real.mean():.1f}, FS chance {dec.fs_rev.mean():.1f}, "
      f"HD {dec.hd_real.mean():.1f}, HD chance {dec.hd_rev.mean():.1f}")
show(signrank(dec.fs_real, dec.fs_rev), "FS error vs chance, per session:")
r, p = stats.pearsonr(dec.n_fs, dec.fs_real)
print(f"Pearson r between number of FS cells and FS median error = {r:.3f}, p = {p:.3g}, n = {len(dec)}")
rs, ps = stats.spearmanr(dec.n_fs, dec.fs_real)
print(f"(Spearman, shown only: rho = {rs:.3f}, p = {ps:.3g})")
fig, ax = plt.subplots(figsize=(4, 3)); ax.scatter(dec.n_fs, dec.fs_real); ax.set_xlabel("FS cells"); ax.set_ylabel("median error (deg)"); plt.show()
print("plotted ranges: cells", dec.n_fs.min(), dec.n_fs.max(), "error", dec.fs_real.min().round(1), dec.fs_real.max().round(1))

**Result: reproduced.** Over 31 sessions the mean of the FS decoder's median error is 36.0 degrees against a chance level of 90.0 (signed-rank over sessions p = 1.2e-6), and the error falls with the number of FS cells (Pearson r = -0.449, p = 0.011, n = 31). The HD-cell decoder's mean median error is 9.9 degrees.

## fig2f-cue-rotation (figshare, data elsewhere)

> In a cue rotation task PoSub-FS cell tuning curves follow the rotated distal landmark
> together with PoSub-HD cells.

The six cue rotation sessions are figshare Dataset_3.tar.gz (claims file), streamed here. Cell
classes are the files' own CellTypes (hd, fs). Methods (Analysis of HD cell realignment after
cue rotation): HD cell spikes in 50 ms bins smoothed with 100 ms s.d.; Bayesian decoding with
tuning curves (1 degree bins, 3 degree smoothing) from the baseline period with a stable cue
(LED1 first interval, as GetBayesianDecoding_CueRotation.m); decoder error the angular
difference between real and decoded HD. Rotations at the starts of the 16 cue epochs
(CueEpochs.mat). Exclusion (criterion): (1) the circular mean decoder error over the 200 s
after the rotation differs by less than 45 degrees from that over the 200 s before it (the
change, as the code computes it, since the cue returns to its first wall every second
rotation); (2) mean speed below 2 cm/s in the preceding epoch (positions from
Tracking_data.csv, metres). Main reading excludes a rotation failing either; the second
reading only one failing both. A sigmoid is fitted to the decoder error (re-centred on the
preceding epoch's mean error) from 50 s before to 200 s after each rotation; realignment ends
where the normalized sigmoid reaches 0.99. For each cell, a tuning curve (1 degree, 3 degree
smoothing, no speed threshold) over the rest of each epoch; its realignment is the lag of
maximum cross-correlation with the curve of the preceding epoch. Per FS cell, the mean
absolute difference between its realignment and the circular mean realignment of the
session's HD cells, over the kept rotations; control, the same with the FS curves from the
time-reversed HD; Wilcoxon signed-rank.

In [ ]:
from scipy.optimize import curve_fit
CU = posub.load_adn(file_id=posub.FIGSHARE_CUE_FILE)
sigm = lambda x, a, b, c, d: a + (b - a) / (1 + 10 ** ((c - x) * d))
cue_rows, rot_rows = [], []
for c in CU:
    t, ang = c["hd_t"], c["hd"]
    hdc = np.where(c["file_types"]["hd"])[0]; fsc = np.where(c["file_types"]["fs"])[0]
    b0, b1 = c["led1"][0]
    mb = posub.in_intervals(t, [(b0, b1)]) & np.isfinite(ang)
    tc_b, _ = posub.tuning_curves([c["spikes"][i] for i in hdc], t, ang, mb)
    wa, wb = c["eps"]["wakeEp"][0]
    edges = np.arange(wa, wb, 0.05); cen = edges[:-1] + 0.025
    Q = gaussian_filter1d(posub.bin_counts([c["spikes"][i] for i in hdc], edges), 2, axis=0)
    dec = np.deg2rad(posub.bayes_decode(tc_b, Q, 0.05))
    j = np.clip(np.searchsorted(t, cen), 1, len(t) - 1); j = np.where(np.abs(t[j - 1] - cen) < np.abs(t[j] - cen), j - 1, j)
    real = np.where(np.abs(t[j] - cen) < 0.02, ang[j], np.nan)
    err = posub.angdiff(real, dec)
    tr = c["track"]; _, uq = np.unique(tr[:, 0], return_index=True); tr = tr[uq]  # repeated timestamps dropped
    tt = tr[:, 0]; xy = tr[:, 1:3] * 100
    sp_ = np.r_[0, np.hypot(*np.diff(gaussian_filter1d(xy, 10, axis=0), axis=0).T) / np.diff(tt)]
    ce = c["cueEp"]
    starts = ce[:, 0]
    cmean = lambda x: np.angle(np.nanmean(np.exp(1j * x[np.isfinite(x)])))
    for k, r0 in enumerate(starts):
        pre = (cen >= r0 - 200) & (cen < r0); post = (cen >= r0) & (cen < r0 + 200)
        change = abs(np.rad2deg(posub.angdiff(cmean(err[post]), cmean(err[pre]))))
        prev_speed = sp_[(tt >= r0 - 200) & (tt < r0)].mean()
        w = (cen >= r0 - 50) & (cen < r0 + 200) & np.isfinite(err)
        x = cen[w]; y = np.rad2deg(posub.angdiff(err[w], cmean(err[pre])))
        try:
            p, _ = curve_fit(sigm, x, y, p0=[np.quantile(y, 0.05), np.quantile(y, 0.95), r0 + 10, 0.1], maxfev=20000)
            yv = sigm(x, *p); yn = (yv - yv.min()) / (yv.max() - yv.min())
            if yv[0] > yv[-1]:
                yn = 1 - yn
            end = x[np.argmin(np.abs(yn - 0.99))]
        except Exception:
            end = np.nan
        # the code's rule (GetSigmoids_CueRotation.m): windows of the last 50 s of each epoch
        w_pre = (cen >= r0 - 50) & (cen < r0); w_post = (cen >= r0 + 150) & (cen < r0 + 200)
        code_change = abs(np.rad2deg(posub.angdiff(cmean(err[w_post]), cmean(err[w_pre]))))
        e_ = err[w_pre][np.isfinite(err[w_pre])]
        code_var = np.rad2deg(1 - np.abs(np.mean(np.exp(1j * e_))))
        mh = (t >= r0 - 50) & (t < r0) & np.isfinite(ang)
        ahv = np.nanmean(np.abs(np.rad2deg(np.diff(np.unwrap(ang[mh]))) / np.diff(t[mh])))
        rot_rows.append(dict(session=c["session"], k=k, change=change, prev_speed=prev_speed, end_s=end - r0 if np.isfinite(end) else np.nan,
                             code_change=code_change, code_var=code_var, ahv=ahv))
    R = pd.DataFrame([r for r in rot_rows if r["session"] == c["session"]])
    # tuning curves over the rest of each epoch (from the end of realignment), and the epoch before the first rotation
    segs = [(starts[0] - 200, starts[0])] + [((r0 + (e if np.isfinite(e) else 0)), ce[k, 1]) for k, (r0, e) in enumerate(zip(starts, R.end_s))]
    cells = np.r_[hdc, fsc]
    H = []; Hr = []
    for a_, b_ in segs:
        m = posub.in_intervals(t, [(a_, b_)]) & np.isfinite(ang)
        H.append(posub.tuning_curves([c["spikes"][i] for i in cells], t, ang, m)[0])
        Hr.append(posub.tuning_curves([c["spikes"][i] for i in cells], t, ang, m, reverse=True)[0])
    nh = len(hdc)
    for k in range(len(starts)):
        lag = posub.circ_xcorr_lag(H[k], H[k + 1]); lag_r = posub.circ_xcorr_lag(Hr[k], Hr[k + 1])
        hd_mean = stats.circmean(np.deg2rad(lag[:nh][np.isfinite(lag[:nh])]))
        for jj, i in enumerate(fsc):
            d = abs(np.rad2deg(posub.angdiff(np.deg2rad(lag[nh + jj]), hd_mean)))
            dr = abs(np.rad2deg(posub.angdiff(np.deg2rad(lag_r[nh + jj]), hd_mean)))
            cue_rows.append(dict(session=c["session"], cell=i, k=k, diff=d, diff_rev=dr,
                                 hd_rot=np.rad2deg(hd_mean) % 360, change=R.change[k], prev_speed=R.prev_speed[k],
                                 code_change=R.code_change[k], code_var=R.code_var[k], ahv=R.ahv[k]))
ROT = pd.DataFrame(rot_rows); CR = pd.DataFrame(cue_rows)
print(ROT.round(1).to_string())
print("rotations failing: change < 45:", int((ROT.change < 45).sum()), "; speed < 2 cm/s:", int((ROT.prev_speed < 2).sum()), "; realignment end (s) median", ROT.end_s.median().round(1), "range", ROT.end_s.min().round(1), ROT.end_s.max().round(1))
for name, excl in (("main: exclude a rotation failing either condition", lambda d: (d.change < 45) | (d.prev_speed < 2)),
                   ("second: exclude only a rotation failing both", lambda d: (d.change < 45) & (d.prev_speed < 2)),
                   ("change condition alone", lambda d: d.change < 45),
                   ("code variant: change < 45, variance > 10 or AHV < 10", lambda d: (d.code_change < 45) | (d.code_var > 10) | (d.ahv < 10))):
    kept = CR[~excl(CR)]
    per = kept.groupby(["session", "cell"])[["diff", "diff_rev"]].mean()
    print(f"{name}: rotations kept {int((~excl(ROT)).sum())} of {len(ROT)}; FS cells {len(per)}")
    show(signrank(per["diff"], per["diff_rev"]), "   FS minus HD realignment |difference| (deg), real vs control:")
print("paper: n = 99 FS cells, z = 8.25")
print("HD rotation (deg) at kept rotations, main reading:", CR[~((CR.change < 45) | (CR.prev_speed < 2))].groupby(["session", "k"]).hd_rot.first().round(0).values)

**Result: reproduced.** In the six figshare cue rotation sessions, the realignment of the 99 FS cells is closer to that of the HD cells than in the time-reversed control (median absolute difference 36.2 against 91.7 degrees, |z| = 8.52, p = 1.6e-17; paper z = 8.25), with 89 of 96 rotations kept under the main exclusion reading, and the same under the other readings and the code's rule (|z| 8.53 to 8.55). The HD cells rotate by about 90 and 270 degrees at the kept rotations.

### Fourier spectra

Each 360-bin tuning curve (3 degree smoothing) is projected on harmonics 1 to 180 (code,
Figure_Fourier). Power is read two ways (claims file): the squared modulus of the complex
coefficient (code) and the root mean square of the sine and cosine projections (Methods'
words). The first ten powers are normalized to their sum.

In [ ]:
H = per_unit("sq3"); H_rev = per_unit("sq3_rev")
FT, FT_rev = posub.fourier(H), posub.fourier(H_rev)
P = {rd: posub.power(FT, rd) for rd in ("sq", "rms")}
P_rev = {rd: posub.power(FT_rev, rd) for rd in ("sq", "rms")}
N10 = {rd: posub.norm10(P[rd]) for rd in P}
N10_rev = {rd: posub.norm10(P_rev[rd]) for rd in P}
fsm, hdm = U.fs.values, U.hd.values
# Parseval check (Extended Data Fig. 3e): 2 x sum of squared moduli = variance of the curve
pv = 2 * P["sq"].sum(0) / H.var(0)
print("Parseval, 2 x sum |c_k|^2 over the curve's variance: median", np.nanmedian(pv[fsm | hdm]).round(4),
      "range", np.nanmin(pv[fsm | hdm]).round(4), np.nanmax(pv[fsm | hdm]).round(4))

## fig3-ten-components-power

> Fourier power of tuning curves decays quickly with frequency, and the first ten components
> hold on average 98% of the total power.

Expected 0.98, tolerance 0.02. Readings: population (a) FS and HD pooled, (b) each alone;
power (a) squared modulus, (b) root mean square. Reproduced when one population reading
under each power reading is within tolerance.

In [ ]:
rows = []
for rd in ("sq", "rms"):
    frac = P[rd][:10].sum(0) / P[rd].sum(0)
    for pop, m in (("FS + HD", fsm | hdm), ("FS", fsm), ("HD", hdm)):
        rows.append(dict(power=rd, population=pop, n=int(m.sum()), mean_fraction=frac[m].mean().round(4),
                         within=abs(frac[m].mean() - 0.98) <= 0.02))
print(pd.DataFrame(rows).to_string())
mp = P["sq"][:, fsm | hdm].mean(1)
print("mean squared-modulus power, harmonics 1 to 12, relative to harmonic 1:", (mp[:12] / mp[0]).round(4))

**Result: partially reproduced.** With power as the squared modulus (the code, and the only reading under which the Parseval check holds, ratio 1.0000), the first ten harmonics hold on average 0.989 of the power of FS and HD cells pooled, 0.969 for FS and 0.994 for HD cells, all within 0.98 +- 0.02, and the mean power falls to 0.10 of the first harmonic's by the fifth. With power as the root mean square of the projections (the Methods' words) the fractions are 0.872, 0.778 and 0.898, none within tolerance, and the criterion asks for one population within tolerance under each power reading.

## fig3g-first-three-components

> Compared with time-reversed controls, the PoSub-FS spectrum has higher power for the first
> three Fourier components only.

Repeated-measures ANOVA, condition (real, time-reversed) by component (1 to 10), within cells;
then a Wilcoxon signed-rank test per component with p multiplied by 10 (criterion; code).

In [ ]:
from statsmodels.stats.anova import AnovaRM
for rd in ("sq", "rms"):
    a, b = N10[rd][:, fsm], N10_rev[rd][:, fsm]
    n = a.shape[1]
    long = pd.DataFrame(dict(cell=np.tile(np.repeat(np.arange(n), 10), 2), comp=np.tile(np.tile(np.arange(1, 11), n), 2),
                             cond=np.repeat(["real", "rev"], 10 * n), y=np.r_[a.T.ravel(), b.T.ravel()]))
    res = AnovaRM(long, "y", "cell", within=["cond", "comp"]).fit().anova_table
    print(f"power {rd}: interaction F({int(res.loc['cond:comp', 'Num DF'])},{int(res.loc['cond:comp', 'Den DF'])}) = "
          f"{res.loc['cond:comp', 'F Value']:.1f}, p = {res.loc['cond:comp', 'Pr > F']:.3g}")
    out = []
    for k in range(10):
        d = signrank(a[k], b[k])
        out.append(dict(component=k + 1, real_median=round(d["median_a"], 4), rev_median=round(d["median_b"], 4),
                        z=round(d["z"], 2), z_over_10=round(d["z"] / 10, 3), p_bonf=min(1, d["p"] * 10),
                        higher=d["median_a"] > d["median_b"]))
    out = pd.DataFrame(out); print(out.to_string())
    sig_hi = out[(out.p_bonf < 0.05) & out.higher].component.tolist()
    sig_hi_mean = [k + 1 for k in range(10) if out.p_bonf[k] < 0.05 and a[k].mean() > b[k].mean()]
    print(f"power {rd}: components significantly higher than control (medians): {sig_hi}; (means): {sig_hi_mean}")
print("paper: F(9,3834) = 64.1, p = 4e-110; components 1 to 3 higher (p = 8e-10, 3e-16, 4e-5)")

**Result: reproduced (resting on an assumption).** With squared-modulus power the condition by component interaction is significant (F(9,3861) = 99.5, p = 1e-167) and, after Bonferroni correction, components 1, 2 and 3 and no others are higher than control (p = 1e-20, 1e-16, 0.028); components 5 to 10 are lower. With root-mean-square power components 1 to 4 are higher. The squared modulus is adopted as the reading the paper's Parseval check supports.

## fig3h-dominant-component

> Of the 427 PoSub-FS cells, 231, 141 and 55 have their highest power in the first, second
> and third Fourier component, proportions similar to the relative mean power of those
> components.

Each FS cell goes to the largest of its first three powers (code, highComp). Readings of
"similar": (a) the proportions are in the same order as the mean relative powers of
components 1 to 3 renormalized over those three; (b) each proportion is within 10% of its
renormalized mean power (read as a relative 10%; an absolute 0.10 is shown too).

In [ ]:
high = np.argmax(N10["sq"][:3], axis=0) + 1
U["high"] = high
cnt = pd.Series(high[fsm]).value_counts().sort_index()
print("FS cells by dominant component (1, 2, 3):", cnt.to_dict(), "of", int(fsm.sum()), "; expected 231 +-23, 141 +-14, 55 +-6")
print("same assignment under the rms reading:", bool((np.argmax(N10["rms"][:3], 0)[fsm] == high[fsm] - 1).all()))
prop = cnt.values / cnt.sum()
for rd in ("sq", "rms"):
    mpw = N10[rd][:3, fsm].mean(1); mpw = mpw / mpw.sum()
    a_ok = list(np.argsort(-prop)) == list(np.argsort(-mpw))
    rel = np.abs(prop - mpw) / mpw
    print(f"power {rd}: proportions {prop.round(3)}, renormalized mean powers {mpw.round(3)}; (a) same order: {a_ok}; "
          f"(b) relative differences {rel.round(3)}, all within 10%: {bool((rel <= 0.10).all())}; absolute differences {np.abs(prop - mpw).round(3)}")
print("per subject, FS cells by dominant component:")
print(U[U.fs].groupby("subject").high.value_counts().unstack(fill_value=0).T.to_string())

**Result: partially reproduced.** FS cells peak in component 1, 2 and 3 in 237, 146 and 47 cases; the first two are within tolerance of 231 and 141, the third is below 55 - 6. The proportions (0.551, 0.340, 0.109) are in the same order as the renormalized mean powers (0.486, 0.326, 0.187), reading (a), but differ by more than 10% for components 1 and 3 (13% and 42%), reading (b), under both power readings.

## ed3g-fs-spectra-heterogeneous

> Fourier signatures are far more variable among PoSub-FS cells than among HD cells of the
> same region, so individual FS cells are narrowly tuned in Fourier space.

Distance of each cell's ten-component spectrum from its own population's mean: Euclidean
(code, ftDist) and KL divergence D_KL(cell || mean) (legend, Methods direction), both power
readings. FS against PoSub-HD, Mann-Whitney U, p multiplied by 3. The ANOVA and the ADN-HD
comparisons are in the ADN section below.

In [ ]:
DIST = {}
for rd in ("sq", "rms"):
    for meas in ("euclid", "kl"):
        d = np.full(len(U), np.nan)
        for m in (fsm, hdm):
            mu = N10[rd][:, m].mean(1, keepdims=True)
            d[m] = np.sqrt(((N10[rd][:, m] - mu) ** 2).sum(0)) if meas == "euclid" else posub.kl(N10[rd][:, m], mu)
        DIST[(rd, meas)] = d
        r = ranksum(d[fsm], d[hdm]); r["p_x3"] = min(1, r["p"] * 3); r["z_over_3"] = r["z"] / 3
        show(r, f"power {rd}, {meas}: FS vs PoSub-HD")
print("paper: legend z = 5.71 (a third of the test's z)")

## ed3l-fs-phases-independent

> Phases of the Fourier components are correlated with each other in individual HD cells but
> random relative to each other in PoSub-FS cells.

Circular correlation (circ_corrcc) across cells between twice the first phase (mod 2 pi) and
the second phase, and three times the first and the third (criterion; code).

In [ ]:
ph = np.angle(FT[:3])
for name, m in (("PoSub-HD", hdm), ("PoSub-FS", fsm)):
    for k in (2, 3):
        rho, p = posub.circ_corrcc(np.mod(k * ph[0, m], 2 * np.pi), np.mod(ph[k - 1, m], 2 * np.pi))
        print(f"{name}: {k} x phase 1 against phase {k}: rho = {rho:.3f}, p = {p:.3g}, n = {int(m.sum())}")
# the FS result with code classes
exc_, fsc_ = classify(U, 0.35, True)
for k in (2, 3):
    rho, p = posub.circ_corrcc(np.mod(k * ph[0, fsc_.values], 2 * np.pi), np.mod(ph[k - 1, fsc_.values], 2 * np.pi))
    print(f"PoSub-FS, code classes: {k} x phase 1 against phase {k}: rho = {rho:.3f}, p = {p:.3g}")

**Result: reproduced.** Across PoSub-HD cells twice the first phase is correlated with the second (rho = 0.32, p < 1e-15) and three times the first with the third (rho = 0.087, p = 1e-4); across FS cells neither is (rho = 0.014, p = 0.77; rho = 0.005, p = 0.91), nor with the code's classes.

## ed4a-autocorrelogram-symmetries

> Tuning curve auto-correlograms of FS cells with the highest power in the second component
> peak at 180 degrees, and those with the highest power in the third at 120 and 240 degrees,
> above controls.

Cross-validated auto-correlograms from the two halves (3 degree smoothing), value at 180
degrees for twofold cells and at 120 degrees for threefold cells, against the same value from
the time-reversed halves; Wilcoxon signed-rank (legend), Mann-Whitney U (code) as a variant.
240 degrees is shown and not judged.

In [ ]:
AC = posub.xval_autocorr(per_unit("h1_3"), per_unit("h2_3"))
AC_rev = posub.xval_autocorr(per_unit("h1_3_rev"), per_unit("h2_3_rev"))
for k, lags in ((2, (180,)), (3, (120, 240))):
    m = fsm & (high == k)
    for lag in lags:
        show(signrank(AC[lag, m], AC_rev[lag, m]), f"FS, dominant component {k}, {lag} deg, signed-rank:")
        show(ranksum(AC[lag, m], AC_rev[lag, m]), f"FS, dominant component {k}, {lag} deg, rank-sum:")
m1 = fsm & (high == 1)
print("for comparison, onefold FS cells at 180 deg: median", np.median(AC[180, m1]).round(3), "control", np.median(AC_rev[180, m1]).round(3))

**Result: reproduced.** The cross-validated auto-correlogram at 180 degrees of the 146 twofold FS cells is above control (median 0.19 against 0.01, signed-rank p = 2e-13; rank-sum p = 2e-14), and at 120 degrees for the 47 threefold cells (0.17 against 0.02, p = 0.0039; rank-sum p = 0.0014). At 240 degrees, shown only, 0.18 against 0.01.

## ADN recordings (figshare, claims with data elsewhere)

The ADN-HD cells of Fig. 3 are not in the dandiset. The claims file names their source, the
paper's figshare item doi:10.6084/m9.figshare.24921252 version 3, Dataset_5.tar.gz. It is
streamed and read in memory (helper `load_adn`). Tuning curves are computed from the spike
times, head direction and the file's speed as for PoSub; ADN-HD cells are units with rate
above 0.5 Hz and HD information at least 0.2 bits per spike (code, GetCellTypes_ADN), among
units the file's BrainArea marks as ADN (code, Figure_Fourier isAdn).

In [ ]:
A = posub.load_adn()
print("ADN sessions:", len(A), "; units:", sum(len(a["spikes"]) for a in A), "; marked ADN:", sum(int(a["adn"].sum()) for a in A))
ADN = []
for a in A:
    d = posub.adn_tuning(a)
    d10 = posub.adn_tuning(a, speed_th=10.0)
    hd = a["adn"] & (d["rate"] > 0.5) & (d["info"] >= 0.2)
    d.update(hd=hd, h10=d10["h"], file_hd=a["file_hd"], session=a["session"])
    ADN.append(d)
adn_hd = np.concatenate([d["hd"] for d in ADN]); adn_file = np.concatenate([d["file_hd"] for d in ADN])
print("ADN-HD cells recomputed:", int(adn_hd.sum()), "; in the files' CellTypes:", int(adn_file.sum()),
      "; agreement:", (adn_hd == adn_file).mean().round(3), "; paper: 97")
HA = np.concatenate([d["h"][:, d["hd"]] for d in ADN], 1)
HA_rev = np.concatenate([d["h_rev"][:, d["hd"]] for d in ADN], 1)
HA10 = np.concatenate([d["h10"][:, d["hd"]] for d in ADN], 1)
HA1 = np.concatenate([d["h1"][:, d["hd"]] for d in ADN], 1); HA2 = np.concatenate([d["h2"][:, d["hd"]] for d in ADN], 1)
adn_sess = np.concatenate([[d["session"]] * int(d["hd"].sum()) for d in ADN])
# contamination scores, ADN-HD
adn_cq = np.concatenate([[posub.contamination(sp) for sp, k in zip(a["spikes"], d["hd"]) if k] for a, d in zip(A, ADN)])

## fig3c-adn-broader

> ADN-HD cells have broader tuning curves than PoSub-HD cells.

Width as the code computes it (full width at half maximum of the peak-aligned, normalized
curve, its two sides averaged); Mann-Whitney U, PoSub-HD against ADN-HD.

In [ ]:
w_pos, off_pos = posub.peak_width(H[:, hdm])
w_adn, off_adn = posub.peak_width(HA)
show(ranksum(w_adn, w_pos), "width (deg), ADN-HD vs PoSub-HD:")
print("paper: n = 1602 and 97, z = 11.4")
print("ADN-HD median width per session:", pd.Series(w_adn).groupby(adn_sess).median().to_dict())

**Result: reproduced.** ADN-HD tuning curves (97 cells, the figshare count) are wider than PoSub-HD curves (median width 62 against 42 degrees, z = 11.58, p = 4e-31; paper z = 11.4).

## fig3e-fourier-signatures-differ

> The average Fourier signatures of PoSub-FS, PoSub-HD and ADN-HD populations differ, with
> ADN-HD cells having more power at low and less at high components than PoSub-HD and
> PoSub-FS cells, which are alike.

Repeated-measures ANOVA of the ten normalized powers, component by cell type interaction,
cell type as a numeric covariate (code: FS 1, PoSub-HD 2, ADN-HD 3) and, as a variant, as a
categorical factor. Direction readings (a) ADN mean above both PoSub means at component 1 and
below both summed over components 3 to 10; (b) ADN cumulative signature above both at every
component 1 to 9. Both power readings.

In [ ]:
FTA = posub.fourier(HA)
NA = {rd: posub.norm10(posub.power(FTA, rd)) for rd in ("sq", "rms")}
for rd in ("sq", "rms"):
    Y = np.r_[N10[rd][:, fsm].T, N10[rd][:, hdm].T, NA[rd].T]
    g = np.r_[np.ones(fsm.sum()), 2 * np.ones(hdm.sum()), 3 * np.ones(NA[rd].shape[1])]
    F, d1, d2, p = posub.rm_interaction(Y, g[:, None])
    Fc, c1, c2, pc = posub.rm_interaction(Y, np.c_[g == 2, g == 3].astype(float))
    Y2 = Y[g < 3]; F2, e1, e2, p2 = posub.rm_interaction(Y2, g[g < 3][:, None])
    mf, mh, ma = N10[rd][:, fsm].mean(1), N10[rd][:, hdm].mean(1), NA[rd].mean(1)
    ra = ma[0] > max(mf[0], mh[0]) and ma[2:].sum() < min(mf[2:].sum(), mh[2:].sum())
    rb = bool(np.all(np.cumsum(ma)[:9] > np.maximum(np.cumsum(mf)[:9], np.cumsum(mh)[:9])))
    print(f"power {rd}: numeric covariate F({d1},{d2}) = {F:.2f}, p = {p:.3g}; categorical F({c1},{c2}) = {Fc:.2f}, p = {pc:.3g}; "
          f"PoSub-FS vs PoSub-HD only F({e1},{e2}) = {F2:.2f}, p = {p2:.3g}")
    print(f"   mean signatures FS {mf.round(3)}\n   PoSub-HD {mh.round(3)}\n   ADN-HD {ma.round(3)}")
    print(f"   reading (a) {ra}; reading (b) {rb}")
print("paper: F(9,19116) = 2.76, p = 0.003")

**Result: not reproduced.** Under the main reading (squared-modulus power, cell type as the code's numeric covariate) the component by cell type interaction is not significant (F(9,18333) = 1.09, p = 0.365; paper F(9,19116) = 2.76, p = 0.003), though the ADN-HD signature has more power at component 1 and less over components 3 to 10 than both PoSub signatures, and its cumulative signature is above both at every component (both direction readings hold under both power readings). The interaction is significant with root-mean-square power (F = 10.68, p = 1e-16) and with cell type as a categorical factor (F(18,18324) = 24.7, p = 5e-82 and 44.6, p = 2e-155). The outcome rests on the choice of power, which the paper's Parseval check settles for the squared modulus, and on the numeric coding, which reproduces the paper's degrees of freedom.

## fig3f-posub-hd-closer-to-fs

> Individual PoSub-HD cell Fourier spectra are closer to the average PoSub-FS signature than
> ADN-HD cell spectra are.

KL divergence between each HD cell's ten-component spectrum and the mean PoSub-FS spectrum,
both directions and both power readings; Mann-Whitney U, PoSub-HD against ADN-HD.

In [ ]:
def fs_distance(rd, Npos, Nadn, Nfs, direction):
    S_ = Nfs.mean(1, keepdims=True)
    f = (lambda c: posub.kl(c, S_)) if direction == "cell||FS" else (lambda c: posub.kl(np.broadcast_to(S_, c.shape), c))
    return f(Npos), f(Nadn)

F3F = {}
for rd in ("sq", "rms"):
    for direction in ("cell||FS", "FS||cell"):
        dp, da = fs_distance(rd, N10[rd][:, hdm], NA[rd], N10[rd][:, fsm], direction)
        r = ranksum(dp, da); F3F[(rd, direction)] = r
        show(r, f"power {rd}, D_KL({direction}): PoSub-HD vs ADN-HD")
print("paper: p < 1e-9, n = 1602 and 97")

**Result: partially reproduced.** PoSub-HD spectra are closer to the mean FS spectrum than ADN-HD spectra under all four readings (median KL 0.082 against 0.133, 0.106 against 0.233, 0.050 against 0.090 and 0.063 against 0.125), with p = 0.008 (squared modulus, D_KL(cell||FS), the Methods' direction), 1e-7, 1.8e-9 and 3e-12. The p-value is below 1e-9 under one reading only.

## ed3k-robust-to-controls

> The smaller distance of PoSub-HD than ADN-HD spectra from the PoSub-FS signature holds after
> excluding multipeaked HD cells, after excluding noisy clusters, and with a 10 cm/s speed
> threshold; ADN-HD cells have higher multipeak scores than PoSub-HD cells.

The Fig. 3f test under each control and each reading. Multipeak score below 0.05 (HD cells;
code), contamination score at most 0.2 (all cells, as the code's onlyGoodClusters filters HD,
FS and viable cells), curves at speed above 10 cm/s (all cells).

In [ ]:
U["cq"] = np.nan
for si, s in enumerate(S):
    idx = U.index[(U.sess == si) & (U.hd | U.fs)]
    for j in idx:
        U.loc[j, "cq"] = posub.contamination(s["spikes"][U.unit[j]])
cq_pos = U.cq.values
print("contamination score above 0.2: PoSub-HD", int((cq_pos[hdm] > 0.2).sum()), "of", int(hdm.sum()),
      "; FS", int((cq_pos[fsm] > 0.2).sum()), "of", int(fsm.sum()), "; ADN-HD", int((adn_cq > 0.2).sum()), "of", len(adn_cq))
_, off_pos = posub.peak_width(H[:, hdm]); _, off_adn = posub.peak_width(HA)
H10 = per_unit("sq3_v10")
rows = []
for rd in ("sq", "rms"):
    N10_v = posub.norm10(posub.power(posub.fourier(H10), rd)); NA_v = posub.norm10(posub.power(posub.fourier(HA10), rd))
    for direction in ("cell||FS", "FS||cell"):
        # multipeak
        dp, da = fs_distance(rd, N10[rd][:, hdm][:, off_pos < 0.05], NA[rd][:, off_adn < 0.05], N10[rd][:, fsm], direction)
        rows.append(dict(control="multipeak < 0.05", power=rd, direction=direction, **ranksum(dp, da)))
        good_f = fsm & (cq_pos <= 0.2)
        dp, da = fs_distance(rd, N10[rd][:, hdm & (cq_pos <= 0.2)], NA[rd][:, adn_cq <= 0.2], N10[rd][:, good_f], direction)
        rows.append(dict(control="contamination <= 0.2", power=rd, direction=direction, **ranksum(dp, da)))
        dp, da = fs_distance(rd, N10_v[:, hdm], NA_v, N10_v[:, fsm], direction)
        rows.append(dict(control="speed > 10 cm/s", power=rd, direction=direction, **ranksum(dp, da)))
K = pd.DataFrame(rows)
print(K.to_string())
show(ranksum(off_adn, off_pos), "multipeak score, ADN-HD vs PoSub-HD:")
print("cells with no width (no side below half maximum): PoSub-HD", int(np.isnan(off_pos).sum()), "ADN-HD", int(np.isnan(off_adn).sum()))
print("paper: multipeak n = 789 and 69, z = 10.4; contamination n = 1472 and 83, z = 5.91; speed n = 1602 and 97, z = 6.46; multipeak scores z = 4.25")

**Result: partially reproduced.** PoSub-HD spectra stay closer to the mean FS spectrum than ADN-HD spectra with multipeaked cells excluded (762 and 69 cells; paper 789 and 69), with noisy clusters excluded (1,399 and 81) and at 10 cm/s, under all four readings (p from 0.019 to 5e-22). The multipeak scores differ (z = 4.18, p = 3e-5; paper z = 4.25) but ADN-HD cells score lower than PoSub-HD cells (median 0.022 against 0.049), the reverse of the claim's direction; the paper's own counts of cells kept below 0.05 (69 of 97 ADN-HD, 789 of 1,602 PoSub-HD) also imply lower ADN-HD scores. A question is asked of a reader.

### ed3g-fs-spectra-heterogeneous, ADN part (reported, not judged)

In [ ]:
for rd in ("sq", "rms"):
    ma = NA[rd].mean(1, keepdims=True)
    da_e = np.sqrt(((NA[rd] - ma) ** 2).sum(0)); da_k = posub.kl(NA[rd], ma)
    for meas, da in (("euclid", da_e), ("kl", da_k)):
        d = DIST[(rd, meas)]
        F1 = stats.f_oneway(d[fsm], d[hdm], da)
        r1, r2 = ranksum(d[fsm], da), ranksum(d[hdm], da)
        print(f"power {rd}, {meas}: one-way ANOVA F(2,{fsm.sum() + hdm.sum() + len(da) - 3}) = {F1.statistic:.1f}, p = {F1.pvalue:.3g}; "
              f"FS vs ADN z/3 = {r1['z'] / 3:.2f}, p x 3 = {min(1, 3 * r1['p']):.3g}; PoSub-HD vs ADN z/3 = {r2['z'] / 3:.2f}, p x 3 = {min(1, 3 * r2['p']):.3g}")
print("paper: F(2,2123) = 169, p = 5e-69; z = 3.40 and 0.45")

**Result: reproduced.** FS spectra are farther from their population mean than PoSub-HD spectra under all four readings (z/3 from 5.78 to 5.97, p x 3 below 1e-66; paper z = 5.71). With the figshare ADN-HD cells the one-way ANOVA gives F(2,2036) = 184 (Euclidean, squared modulus; paper F(2,2123) = 169), FS against ADN z/3 = 3.42 and PoSub-HD against ADN z/3 = 0.48 (paper 3.40 and 0.45), reported and not judged.

## ed4cd-isomap-posub-central

> In a joint Isomap embedding of tuning curve auto-correlograms, PoSub-HD cells lie nearer the
> centre of the PoSub-FS distribution than ADN-HD cells.

Isomap (12 neighbours, 2 dimensions) on the cross-validated auto-correlograms of FS,
PoSub-HD and ADN-HD cells together; Euclidean distance of each HD cell to the mean FS
position; Mann-Whitney U (criterion; Methods). Isomap from scikit-learn, not the MATLAB
toolbox the paper used.

In [ ]:
from sklearn.manifold import Isomap
ACA = posub.xval_autocorr(HA1, HA2)
Xac = np.c_[AC[:, fsm], AC[:, hdm], ACA].T
grp = np.r_[np.zeros(fsm.sum()), np.ones(hdm.sum()), 2 * np.ones(ACA.shape[1])]
okr = np.isfinite(Xac).all(1)
print("cells dropped for an undefined auto-correlogram (flat half):", int((~okr).sum()), "by group", np.bincount(grp[~okr].astype(int), minlength=3))
emb = Isomap(n_neighbors=12, n_components=2).fit_transform(Xac[okr]); g = grp[okr]
c = emb[g == 0].mean(0); dist = np.linalg.norm(emb - c, axis=1)
show(ranksum(dist[g == 1], dist[g == 2]), "distance to FS centre, PoSub-HD vs ADN-HD:")
print("paper: n = 1602 and 97, z = 4.26")
fig, ax = plt.subplots(figsize=(4.5, 4))
for k, col in ((0, "tab:orange"), (1, "tab:purple"), (2, "magenta")):
    ax.scatter(*emb[g == k].T, s=3, c=col, label=["FS", "PoSub-HD", "ADN-HD"][k])
ax.legend(); plt.show()
print("embedding range", emb.min(0).round(2), emb.max(0).round(2))

**Result: reproduced.** In the joint Isomap embedding of auto-correlograms, PoSub-HD cells lie nearer the mean FS position than ADN-HD cells (median distance 2.34 against 3.39, z = -2.93, p = 0.0034, n = 1,512 and 97; paper z = 4.26).

## fig4f-synaptic-weights-dispersed

> Putative synaptic weights from PoSub-HD to PoSub-FS cells, estimated from spike train
> cross-correlograms, are heavy-tailed with an s.d. larger than their mean, in the range where
> simulations predict the radial symmetries of FS tuning.

Cross-correlograms over the whole recording in 0.2 ms bins, HD cell as reference; baseline
from a partially hollowed Gaussian (cch_conv, 41 bins); the 99th percentile of the Poisson at
the largest predicted count within 0 to 2 ms; at least two consecutive bins above it; peak at
positive lag, latency above 0.5 ms, no peak at negative lag (criterion; GetSynapticPairs.m,
FindSynapse_Exc, Figure_SynapticConnections.m). Weight strZ (peak minus prediction over the
square root of the prediction); strR is a variant. CCG span plus or minus 50 ms (Methods) and
plus or minus 25 ms (code).

In [ ]:
t_syn = time.time()
syn_rows = []
for si, s in enumerate(S):
    u = U[U.sess == si]
    hd_idx = np.where(u.hd.values)[0]; fs_idx = np.where(u.fs.values)[0]
    for i in hd_idx:
        for j in fs_idx:
            row = dict(subject=s["subject"], hd=i, fs=j)
            for span, nb in (("50", 500), ("25", 250)):
                c = posub.ccg_counts(s["spikes"][i], s["spikes"][j], nbins=nb)
                ln, zn, lp, zp, rp = posub.find_synapse_exc(c)
                row.update({f"lat_neg{span}": ln, f"z_neg{span}": zn, f"lat{span}": lp, f"z{span}": zp, f"r{span}": rp})
            syn_rows.append(row)
SY = pd.DataFrame(syn_rows)
print(f"HD-FS pairs: {len(SY)}; CCGs in {time.time() - t_syn:.0f} s")
for span in ("50", "25"):
    syn = SY[SY[f"z{span}"].notna() & SY[f"z_neg{span}"].isna() & (SY[f"lat{span}"] > 0.5)]
    for w in ("z", "r"):
        x = syn[f"{w}{span}"]
        print(f"span +-{span} ms, weight str{w.upper()}: {len(syn)} connections ({len(syn) / len(SY):.3%} of pairs); mean {x.mean():.2f}, "
              f"s.d. {x.std():.2f}, CV {x.std() / x.mean():.2f}, skewness {stats.skew(x):.2f}, median {x.median():.2f}, max {x.max():.2f}")
    print("   connections per subject:", syn.groupby("subject").size().to_dict())
fig, ax = plt.subplots(figsize=(4, 3))
x = SY[SY.z50.notna() & SY.z_neg50.isna() & (SY.lat50 > 0.5)].z50
ax.hist(x, 40); ax.set_xlabel("strZ"); plt.show()
print("plotted range:", x.min().round(2), x.max().round(2))

**Result: reproduced.** 1,176 putative HD-to-FS connections are found among 24,498 pairs (4.8%). Their strZ weights have mean 8.58 and s.d. 16.32 (CV 1.90) and skewness 9.34, heavy-tailed with an s.d. above the mean; the CCG span (50 or 25 ms) gives the same connections. The strR weights, a variant, have CV 0.71.

## Optogenetic sessions

Sessions with TRN injections in VGAT-Cre mice and light over ADN: ArchT (virus field names
ArchT and the thalamic reticular nucleus) and control (GFP only). Light ON is each 1 s pulse,
OFF the 4 s before each onset (Methods; code offStart = 4). Cells are the square-epoch classes
above, kept when their rates in ON, OFF and the square epoch (first to last tracked sample)
all exceed 0.5 Hz (code, Figure_Optogenetics). Tuning curves in ON and OFF use no speed
threshold (Methods): 1 degree bins with 3 degree smoothing for widths, 6 degree bins without
smoothing for gain.

In [ ]:
def opto_group(s):
    v = s["virus"]
    if "thalamic reticular" not in v:
        return None
    return "ArchT" if "ArchT" in v else ("control" if "GFP" in v else None)

OP = []
for si, s in enumerate(S):
    grp = opto_group(s)
    if grp is None or "pulses" not in s:
        continue
    on = s["pulses"][:240]; off = np.c_[on[:, 0] - 4, on[:, 0]]
    t, ang = s["hd_t"], s["hd"]
    ba, bb = posub.tracked_epoch(s, "wake_square")
    m_on = posub.in_intervals(t, on) & np.isfinite(ang); m_off = posub.in_intervals(t, off) & np.isfinite(ang)
    h_on, _ = posub.tuning_curves(s["spikes"], t, ang, m_on); h_off, _ = posub.tuning_curves(s["spikes"], t, ang, m_off)
    g_on, _ = posub.tuning_curves(s["spikes"], t, ang, m_on, nbins=60, sd=0); g_off, _ = posub.tuning_curves(s["spikes"], t, ang, m_off, nbins=60, sd=0)
    w_on, _ = posub.peak_width(h_on); w_off, _ = posub.peak_width(h_off)
    u = U[U.sess == si]
    for i, sp in enumerate(s["spikes"]):
        fr_on = np.array([((sp >= a) & (sp < b)).sum() / (b - a) for a, b in on])
        fr_off = np.array([((sp >= a) & (sp < b)).sum() / (b - a) for a, b in off])
        base = ((sp >= ba) & (sp < bb)).sum() / (bb - ba)
        x, y = g_off[:, i], g_on[:, i]
        mx = x.max()
        ok = mx > 0 and x.std() > 0 and y.std() > 0
        r, p = stats.pearsonr(y, x) if ok else (np.nan, np.nan)
        slope, icpt = np.polyfit(x / mx, y / mx, 1) if ok else (np.nan, np.nan)
        OP.append(dict(subject=s["subject"], group=grp, sess=si, unit=i, hd=bool(u.hd.iloc[i]), fs=bool(u.fs.iloc[i]),
                       rate_on=fr_on.mean(), rate_off=fr_off.mean(), rate_base=base,
                       p_light=stats.wilcoxon(fr_on, fr_off).pvalue if np.any(fr_on != fr_off) else 1.0,
                       r_tc=r, p_tc=p, mult=slope, add=icpt, w_on=w_on[i], w_off=w_off[i], g_off=x, g_on=y))
OP = pd.DataFrame(OP)
OP["gd"] = (OP.rate_on > 0.5) & (OP.rate_off > 0.5) & (OP.rate_base > 0.5)
OP["mod"] = OP.rate_on / OP.rate_off
print("opto sessions:", OP.groupby("group").subject.unique().to_dict())
print("cells kept, by group and class:")
print(OP[OP.gd].groupby("group")[["hd", "fs"]].sum().to_string())
print("paper: ArchT 83 HD and 47 FS cells (5 mice); control 89 HD and 38 FS (3 mice)")
print("light pulses per session:", {S[si]["subject"]: len(S[si]["pulses"]) for si in OP.sess.unique()})
# the light times share the spike clock: the ON/OFF ratio with the true pulse times against the
# same ratio with every window moved 2.5 s later (ON in the middle of the 4 s gap, OFF across the
# pulse and the gap), over all kept cells
for grp in ("ArchT", "control"):
    d = OP[OP.gd & (OP.group == grp)]
    sh = []
    for si in d.sess.unique():
        s = S[si]; on = s["pulses"][:240] + 2.5; off = np.c_[on[:, 0] - 4, on[:, 0]]
        for i in d[d.sess == si].unit:
            sp = s["spikes"][i]
            ron = posub.in_intervals(sp, on).sum() / 240; roff = posub.in_intervals(sp, off).sum() / 960
            sh.append(ron / roff if roff > 0 else np.nan)
    print(f"{grp}: median ON/OFF rate ratio over {len(d)} kept cells {d['mod'].median():.3f}; with windows moved 2.5 s {np.nanmedian(sh):.3f}")

## fig5b-posub-rates-increase

> Thalamic disinhibition raises the firing rates of PoSub-HD and PoSub-FS cells in ArchT
> mice; the rise of FS cells is larger than in control-virus mice.

In [ ]:
for cls in ("fs", "hd"):
    a = OP[OP.gd & OP[cls] & (OP.group == "ArchT")]; c = OP[OP.gd & OP[cls] & (OP.group == "control")]
    show(ranksum(a["mod"], c["mod"]), f"{cls.upper()} ON/OFF ratio, ArchT vs control:")
    show(signrank(a.rate_on, a.rate_off), f"{cls.upper()} ArchT, ON vs OFF rate per cell:")
    print("   by ArchT subject, median ratio:", a.groupby("subject")["mod"].median().round(3).to_dict(),
          "; control:", c.groupby("subject")["mod"].median().round(3).to_dict())
print("paper: FS n = 47 and 38, z = 4.99; HD n = 83 and 89, z = 1.69")

**Result: reproduced.** In the dandiset's two ArchT mice, FS and HD cells raise their rates with light (median ON/OFF ratio 1.33 and 1.46; signed-rank p = 8e-6 and 3e-9), above the three control mice (1.04 and 1.00); the FS rise exceeds control (z = 5.41, p = 6e-8, n = 33 and 43; paper z = 4.99 for 47 and 38). The HD comparison with control is also significant here (z = 7.77), where the paper's is not (z = 1.69). The dandiset holds 2 of the paper's 5 ArchT mice.

## ed7gj-tuning-preserved

> Thalamic disinhibition preserves the HD tuning of PoSub-HD and PoSub-FS cells.

Two claims of absence on the legends' tests: PoSub-HD width OFF against ON (ArchT),
Wilcoxon signed-rank (rank-sum, the code's test, as a variant); PoSub-FS ON-OFF tuning curve
correlation (6 degree bins, unsmoothed), ArchT against control, Mann-Whitney U.

In [ ]:
a = OP[OP.gd & OP.hd & (OP.group == "ArchT")]
show(signrank(a.w_off, a.w_on), "PoSub-HD width (deg), OFF vs ON, signed-rank:")
show(ranksum(a.w_off, a.w_on), "PoSub-HD width, OFF vs ON, rank-sum (code):")
af = OP[OP.gd & OP.fs & (OP.group == "ArchT")]; cf = OP[OP.gd & OP.fs & (OP.group == "control")]
show(ranksum(af.r_tc, cf.r_tc), "PoSub-FS ON-OFF tuning correlation, ArchT vs control:")
print("paper: width z = 0.30 (n read as 83); FS correlation n = 47 and 38, z = 1.14")

**Result: reproduced.** Light does not change PoSub-HD tuning widths (44 against 46 degrees, signed-rank p = 0.53, n = 62; rank-sum p = 1.0), and FS ON-OFF tuning correlations do not differ between ArchT and control (median 0.72 against 0.69, p = 0.68, n = 33 and 43).

## fig5e-hd-multiplicative-fs-additive

> Thalamic gain modulates PoSub-HD cell tuning multiplicatively and PoSub-FS cell tuning
> additively, so that FS cells receive thalamic input that is uniform across directions.

ON and OFF curves (6 degree bins, unsmoothed, no speed threshold) divided by the OFF maximum;
linear fit of ON on OFF, slope multiplicative and intercept additive factor. ArchT cells are
excluded (a) when both light modulation (Wilcoxon signed-rank of per-pulse ON against OFF
rates) and the ON-OFF tuning correlation are not significant, (b) when either is not, (c)
when the correlation is not (code).

In [ ]:
sig_l, sig_r = OP.p_light < 0.05, OP.p_tc < 0.05
keep = {"(a) exclude when both fail": sig_l | sig_r, "(b) exclude when either fails": sig_l & sig_r, "(c) correlation only (code)": sig_r}
G = {}
for name, k in keep.items():
    ah = OP[OP.gd & OP.hd & (OP.group == "ArchT") & k]; afs = OP[OP.gd & OP.fs & (OP.group == "ArchT") & k]
    ch = OP[OP.gd & OP.hd & (OP.group == "control")]; cfs = OP[OP.gd & OP.fs & (OP.group == "control")]
    excl = (int((OP.gd & OP.hd & (OP.group == "ArchT")).sum() - len(ah)), int((OP.gd & OP.fs & (OP.group == "ArchT")).sum() - len(afs)))
    print(f"{name}: ArchT cells excluded HD {excl[0]}, FS {excl[1]}")
    res = {"HD vs FS, mult": ranksum(ah.mult, afs.mult), "HD vs FS, add": ranksum(ah["add"], afs["add"]),
           "HD vs control, mult": ranksum(ah.mult, ch.mult), "HD vs control, add": ranksum(ah["add"], ch["add"]),
           "FS vs control, mult": ranksum(afs.mult, cfs.mult), "FS vs control, add": ranksum(afs["add"], cfs["add"])}
    for k2, v in res.items():
        show(v, "   " + k2 + ":")
    G[name] = res
print("paper: HD vs FS additive z = 7.72, multiplicative z = 5.99; HD vs control additive z = 1.57, multiplicative z = 6.80; FS vs control additive z = 4.16, multiplicative z = 0.36")

**Result: partially reproduced.** Under all three exclusion readings HD multiplicative factors exceed FS ones (p from 1e-5 to 4e-8) and FS additive factors exceed HD ones (p from 5e-7 to 3e-10), HD multiplicative and FS additive factors exceed control (p below 3e-7 and 4e-4), and FS multiplicative factors do not differ from control (p 0.21 to 0.40). HD additive factors do not differ from control under readings (a) and (c) (p = 0.19) but do under reading (b) (p = 0.015), which excludes 31 of 62 HD cells; the paper excluded 1 FS cell and no HD cell, as readings (a) and (c) do here (0 HD and 0 or 1 FS cells).

## ed8cd-multiplicative-detectable

> Simulations of noisy gain on real FS tuning curves show that multiplicative gain would be
> detectable at the FS cells' tuning correlations, and the FS data are best explained by
> additive gain.

As Figure_GainSimulation.m: OFF curves (6 degree bins, unsmoothed) of all kept cells; for
ArchT cells the simulated ON curve is (OFF + a x max(OFF)) x m with, per cell, no gain
(m = 1, a = 0), multiplicative gain (m drawn from N(1.2, 0.33), a = 0) or additive gain
(m = 1, a drawn from N(0.2, 0.1)); control cells get ON = OFF. Gaussian noise of s.d.
1.8 x 1.8 Hz (code, noiseMag) is added to OFF and ON independently, and factors are computed
as for the data. The simulated ArchT FS factors are compared with the simulated control FS
factors, Mann-Whitney U. The code's choice of FS cells (ArchT cells with a significant tuning
correlation, reading (c)) is used. Seed fixed.

In [ ]:
rng = np.random.default_rng(8)
fs_a = OP[OP.gd & OP.fs & (OP.group == "ArchT") & (OP.p_tc < 0.05)]
fs_c = OP[OP.gd & OP.fs & (OP.group == "control")]
def sim_factors(rows, mult, add, noise=1.8 * 1.8):
    out = []
    for _, r in rows.iterrows():
        x = r.g_off
        m = rng.normal(*mult) if mult else 1.0
        a = rng.normal(*add) if add else 0.0
        y = (x + a * x.max()) * m
        xo = x + rng.normal(0, noise, len(x)); yo = y + rng.normal(0, noise, len(x))
        mx = xo.max()
        sl, ic = np.polyfit(xo / mx, yo / mx, 1)
        out.append((sl, ic, np.corrcoef(xo, yo)[0, 1]))
    return np.array(out)
ctrl = sim_factors(fs_c, None, None)
print(f"observed ON-OFF correlation of ArchT FS cells: median {fs_a.r_tc.median():.3f}; simulated control: median {np.median(ctrl[:, 2]):.3f}")
for name, mult, add in (("no gain", None, None), ("multiplicative", (1.2, 0.33), None), ("additive", None, (0.2, 0.1))):
    sim = sim_factors(fs_a, mult, add)
    rm_, ra_ = ranksum(sim[:, 0], ctrl[:, 0]), ranksum(sim[:, 1], ctrl[:, 1])
    print(f"{name}: simulated correlation median {np.median(sim[:, 2]):.3f}; multiplicative vs control z = {rm_['z']:.2f}, p = {rm_['p']:.3g} "
          f"(medians {rm_['median_a']:.3f}, {rm_['median_b']:.3f}); additive vs control z = {ra_['z']:.2f}, p = {ra_['p']:.3g} "
          f"(medians {ra_['median_a']:.3f}, {ra_['median_b']:.3f})")
    vm, va = ranksum(sim[:, 0], fs_c.mult), ranksum(sim[:, 1], fs_c["add"])
    print(f"   variant, against the recorded control FS factors: multiplicative p = {vm['p']:.3g}, additive p = {va['p']:.3g}")
r1, r2 = ranksum(fs_a.mult, fs_c.mult), ranksum(fs_a["add"], fs_c["add"])
print(f"data: FS multiplicative vs control p = {r1['p']:.3g}, additive vs control p = {r2['p']:.3g}")
# Methods reading: the noise is titrated until the simulated correlations resemble the observed
# ones. Noise s.d. (Hz) chosen on a grid so that the no-gain simulation's median ON-OFF
# correlation is closest to the observed median of the ArchT FS cells.
grid = np.arange(1.0, 12.01, 0.25)
med = []
for nz in grid:
    rng = np.random.default_rng(80)
    med.append(np.median(sim_factors(fs_a, None, None, noise=nz)[:, 2]))
nz = grid[np.argmin(np.abs(np.array(med) - fs_a.r_tc.median()))]
print(f"titrated noise s.d. {nz:.2f} Hz (no-gain median r {med[int(np.argmin(np.abs(np.array(med) - fs_a.r_tc.median())))]:.3f}, observed {fs_a.r_tc.median():.3f})")
rng = np.random.default_rng(81)
ctrl_t = sim_factors(fs_c, None, None, noise=nz)
for name, mult, add in (("no gain", None, None), ("multiplicative", (1.2, 0.33), None), ("additive", None, (0.2, 0.1))):
    sim = sim_factors(fs_a, mult, add, noise=nz)
    rm_, ra_ = ranksum(sim[:, 0], ctrl_t[:, 0]), ranksum(sim[:, 1], ctrl_t[:, 1])
    vm, va = ranksum(sim[:, 0], fs_c.mult), ranksum(sim[:, 1], fs_c["add"])
    print(f"titrated, {name}: median r {np.median(sim[:, 2]):.3f}; vs simulated control: multiplicative p = {rm_['p']:.3g}, additive p = {ra_['p']:.3g}; "
          f"vs recorded control: multiplicative p = {vm['p']:.3g}, additive p = {va['p']:.3g}")
# how often the criterion holds over 50 seeds (titrated noise, simulated control)
hits = []
for seed in range(50):
    rng = np.random.default_rng(1000 + seed)
    c_ = sim_factors(fs_c, None, None, noise=nz); mu_ = sim_factors(fs_a, (1.2, 0.33), None, noise=nz); ad_ = sim_factors(fs_a, None, (0.2, 0.1), noise=nz)
    hits.append((ranksum(mu_[:, 0], c_[:, 0])["p"] < 0.05, ranksum(ad_[:, 1], c_[:, 1])["p"] < 0.05, ranksum(ad_[:, 0], c_[:, 0])["p"] >= 0.05))
hits = np.array(hits)
print("over 50 seeds: multiplicative detected", hits[:, 0].mean(), "; additive detected", hits[:, 1].mean(),
      "; additive simulation without a multiplicative difference", hits[:, 2].mean(), "; all three", hits.all(1).mean())

**Result: reproduced (resting on an assumption).** With the noise titrated so that the no-gain simulation matches the observed median ON-OFF correlation of ArchT FS cells (4.25 Hz; median r 0.731 against 0.730, the Methods' procedure), simulated multiplicative gain is detected (p = 0.022) and simulated additive gain shows the data's pattern (additive p = 8e-5, multiplicative p = 0.13), as in the data (p = 3e-4 and 0.27); over 50 seeds all three hold in 80%. With the code's fixed noise (3.24 Hz, simulated correlations 0.82), the additive simulation also changes the multiplicative factor (p = 0.008), and against the recorded control factors multiplicative gain is not detected (p = 0.80).

## ADN optogenetic sessions (figshare, claims with data elsewhere)

The ADN recordings with light over ADN are not in the dandiset. The claims file names their
source, figshare doi:10.6084/m9.figshare.24921252 version 3, Dataset_6.tar.gz, streamed here.
The same tarball holds the PoSub ArchT sessions of three mice (A1801, A1804, A1806) that the
dandiset lacks; they are listed and not used, since the PoSub claims are tested on the
dandiset. ON is each of the first 240 pulses (Opto_TS.csv), OFF the 4 s before each onset;
cells are kept with rates above 0.5 Hz in ON, OFF and the square epoch (code,
Figure_Optogenetics). ADN-HD cells as GetCellTypes_ADN (rate above 0.5 Hz, HD information at
least 0.2 bits per spike in the square epoch at speed above 2 cm/s); non-HD cells the others
above 0.5 Hz.

In [ ]:
O = posub.load_adn(file_id=posub.FIGSHARE_OPTO_FILE)
print("sessions in Dataset_6:", [(o["session"], "ADN" if o["adn"].any() else "PoSub", {1: "ArchT", 2: "control"}.get(o.get("group")), len(o["spikes"])) for o in O])
dandi_subjects = {s["subject"] for s in S}
print("PoSub opto sessions in Dataset_6 whose mouse is not in the dandiset:",
      [o["session"] for o in O if not o["adn"].any() and o["session"].split("-")[0] not in dandi_subjects])
AO = []
for o in O:
    if not o["adn"].any():
        continue
    o = dict(o); o["eps"] = dict(o["eps"]); o["eps"].setdefault("wake1Ep", o["eps"]["wakeEp"])
    tun = posub.adn_tuning(o)
    hd = (tun["rate"] > 0.5) & (tun["info"] >= 0.2)
    nhd = (tun["rate"] > 0.5) & (tun["info"] < 0.2)
    on = o["pulses"][:240, :2]; off = np.c_[on[:, 0] - 4, on[:, 0]]
    t, ang = o["hd_t"], o["hd"]
    m_on = posub.in_intervals(t, on) & np.isfinite(ang); m_off = posub.in_intervals(t, off) & np.isfinite(ang)
    h_on, _ = posub.tuning_curves(o["spikes"], t, ang, m_on); h_off, _ = posub.tuning_curves(o["spikes"], t, ang, m_off)
    g_on, _ = posub.tuning_curves(o["spikes"], t, ang, m_on, nbins=60, sd=0); g_off, _ = posub.tuning_curves(o["spikes"], t, ang, m_off, nbins=60, sd=0)
    w_on, _ = posub.peak_width(h_on); w_off, _ = posub.peak_width(h_off)
    ep = o["eps"]["wakeEp"]; tt = t[posub.in_intervals(t, ep) & np.isfinite(ang)]; ba, bb = tt[0], tt[-1]
    for i, sp in enumerate(o["spikes"]):
        fr_on = np.array([((sp >= a) & (sp < b)).sum() / (b - a) for a, b in on])
        fr_off = np.array([((sp >= a) & (sp < b)).sum() / (b - a) for a, b in off])
        x, y = g_off[:, i], g_on[:, i]; mx = x.max()
        ok = mx > 0 and x.std() > 0 and y.std() > 0
        r, p = stats.pearsonr(y, x) if ok else (np.nan, np.nan)
        slope, icpt = np.polyfit(x / mx, y / mx, 1) if ok else (np.nan, np.nan)
        AO.append(dict(session=o["session"], mouse=o["session"].split("-")[0], hd=bool(hd[i]), nhd=bool(nhd[i]), file_hd=bool(o["file_hd"][i]),
                       rate_on=fr_on.mean(), rate_off=fr_off.mean(), rate_base=((sp >= ba) & (sp < bb)).sum() / (bb - ba),
                       p_light=stats.wilcoxon(fr_on, fr_off).pvalue if np.any(fr_on != fr_off) else 1.0, r_tc=r, p_tc=p,
                       mult=slope, add=icpt, w_on=w_on[i], w_off=w_off[i]))
AO = pd.DataFrame(AO)
AO["gd"] = (AO.rate_on > 0.5) & (AO.rate_off > 0.5) & (AO.rate_base > 0.5)
AO["mod"] = AO.rate_on / AO.rate_off
print("ADN opto sessions:", AO.session.nunique(), "mice:", AO.mouse.nunique(), "; units", len(AO), "; kept", int(AO.gd.sum()),
      "; ADN-HD kept", int((AO.gd & AO.hd).sum()), "; non-HD kept", int((AO.gd & AO.nhd).sum()),
      "; agreement of recomputed HD with the files' CellTypes", (AO.hd == AO.file_hd).mean().round(3))
print("paper: 127 thalamic cells including 52 HD cells from 3 mice")

## ed7-adn-disinhibition

> Light over ADN in VGAT-Cre mice with ArchT in TRN raises the firing of ADN-HD cells but not
> of non-HD thalamic cells, without broadening ADN-HD tuning curves.

Modulation ON over OFF rate, ADN-HD against non-HD, Mann-Whitney U, with the ADN-HD median
above 1; ADN-HD width (1 degree bins, 3 degree smoothing) OFF against ON, Wilcoxon signed-rank
(a claim of absence), the code's rank-sum as a variant.

In [ ]:
a = AO[AO.gd & AO.hd]; n_ = AO[AO.gd & AO.nhd]
show(ranksum(a["mod"], n_["mod"]), "ON/OFF ratio, ADN-HD vs non-HD:")
show(signrank(a.rate_on, a.rate_off), "ADN-HD, ON vs OFF rate:")
show(signrank(n_.rate_on, n_.rate_off), "non-HD, ON vs OFF rate:")
show(signrank(a.w_off, a.w_on), "ADN-HD width OFF vs ON, signed-rank:")
show(ranksum(a.w_off, a.w_on), "ADN-HD width OFF vs ON, rank-sum (code):")
print("by mouse, ADN-HD median ratio:", a.groupby("mouse")["mod"].median().round(3).to_dict(), "; non-HD:", n_.groupby("mouse")["mod"].median().round(3).to_dict())
print("paper: n = 52 and 75, z = 6.57; width z = 0.67, p = 0.5")

**Result: reproduced.** In the figshare ADN sessions (10 sessions, 3 mice, 127 cells kept, 53 ADN-HD), light raises ADN-HD rates more than non-HD rates (median ON/OFF ratio 1.54 against 1.06, z = 6.85, p = 8e-12; paper z = 6.57), non-HD rates do not change significantly (p = 0.13), and ADN-HD widths do not change (62 against 58 degrees, signed-rank p = 0.61; rank-sum p = 0.80).

## ed8e-adn-mixed-gain

> ADN-HD cells show multiplicative factors like those of PoSub-HD cells and larger additive
> factors.

Factors as in fig5e-hd-multiplicative-fs-additive; ADN-HD (figshare) against the dandiset's
ArchT PoSub-HD cells, Mann-Whitney U, under the three exclusion readings.

In [ ]:
for name in keep:
    kk = {"(a) exclude when both fail": lambda d: (d.p_light < 0.05) | (d.p_tc < 0.05),
          "(b) exclude when either fails": lambda d: (d.p_light < 0.05) & (d.p_tc < 0.05),
          "(c) correlation only (code)": lambda d: d.p_tc < 0.05}[name]
    ad = AO[AO.gd & AO.hd & kk(AO)]
    ph = OP[OP.gd & OP.hd & (OP.group == "ArchT") & kk(OP)]
    rm_, ra_ = ranksum(ad.mult, ph.mult), ranksum(ad["add"], ph["add"])
    print(f"{name}: ADN-HD n = {len(ad)}, PoSub-HD n = {len(ph)}")
    show(rm_, "   multiplicative, ADN-HD vs PoSub-HD:"); show(ra_, "   additive, ADN-HD vs PoSub-HD:")
print("paper: multiplicative z = 0.55, additive z = 5.07")

**Result: reproduced.** Figshare ADN-HD cells and the dandiset's ArchT PoSub-HD cells have similar multiplicative factors (median 1.26 against 1.30, p = 0.47) and ADN-HD additive factors are larger (0.032 against 0.009, p = 7e-6; paper z = 0.55 and 5.07); the same holds under reading (b) (p = 0.62 and 0.002).

## fig6c-coupling-preserved-wake-rem

> GLM cross-coupling of PoSub-HD pairs, PoSub-FS pairs and PoSub-HD to PoSub-FS pairs is
> preserved between wakefulness and REM sleep, less for FS pairs than for HD pairs.

Sessions with at least 100 s of REM (Methods). As GetGLM_Forward.m and GetGLM_Reverse.m:
spike counts of every unit in 100 ms bins over the whole recording, smoothed with a Gaussian of
1 bin s.d.; the population rate (sum over all units) smoothed alike and z-scored; cell B's
train z-scored; Poisson GLM of cell A's smoothed train on the population rate and cell B,
within WAKE (square epoch) or REM; the coefficient on B at lag 0, averaged with that of the
reversed pair; pairs with a fit that did not converge removed. Cells with rates above 1 Hz in
both WAKE and REM (code, Figure_GLMRemWake). Only lag 0 is fitted (the claim uses lag 0).

In [ ]:
from scipy.ndimage import gaussian_filter1d
glm_rows = []
t_glm = time.time()
for si, s in enumerate(S):
    remiv = s["sleep"].get("rem", np.zeros((0, 2)))
    if (remiv[:, 1] - remiv[:, 0]).sum() < 100:
        continue
    u = U[U.sess == si]
    tmax = max(b for v in s["epochs"].values() for a, b in v)
    edges = np.arange(0, tmax + 0.1, 0.1); cen = edges[:-1] + 0.05
    Q = gaussian_filter1d(posub.bin_counts(s["spikes"], edges), 1, axis=0)
    mua = Q.sum(1); mua = (mua - mua.mean()) / mua.std()
    Qz = (Q - Q.mean(0)) / Q.std(0)
    wa, wb = s["epochs"]["wake_square"][0]
    states = {"WAKE": (cen >= wa) & (cen < wb), "REM": posub.in_intervals(cen, remiv)}
    rate_w = np.array([((sp >= wa) & (sp < wb)).sum() / (wb - wa) for sp in s["spikes"]])
    rem_dur = (remiv[:, 1] - remiv[:, 0]).sum()
    rate_r = np.array([posub.in_intervals(sp, remiv).sum() / rem_dur for sp in s["spikes"]])
    cells = np.where((u.hd | u.fs).values & (rate_w > 1) & (rate_r > 1))[0]
    beta, beta_c = {}, {}
    for st, msk in states.items():
        Bm = np.full((len(cells), len(cells)), np.nan); Bc = Bm.copy()
        for ia, a in enumerate(cells):
            others = np.array([c for c in cells if c != a])
            b, ok, conv = posub.poisson_glm_batch(Q[msk, a], mua[msk], Qz[msk][:, others])
            col = [list(cells).index(c) for c in others]
            Bm[ia, col] = np.where(ok, b[2], np.nan); Bc[ia, col] = np.where(conv, b[2], np.nan)
        beta[st], beta_c[st] = Bm, Bc
    cls = np.where(u.hd.values[cells], "HD", "FS")
    tcr = posub.zero_lag_r(np.repeat(TC[si]["sq3"][:, cells], len(cells), 1), np.tile(TC[si]["sq3"][:, cells], (1, len(cells)))).reshape(len(cells), len(cells))
    for i in range(len(cells)):
        for j in range(i + 1, len(cells)):
            pair = "-".join(sorted([cls[i], cls[j]], key=lambda c: c != "HD"))
            glm_rows.append(dict(subject=s["subject"], pair=pair, wake=(beta["WAKE"][i, j] + beta["WAKE"][j, i]) / 2,
                                 rem=(beta["REM"][i, j] + beta["REM"][j, i]) / 2, tc_r=tcr[i, j],
                                 wake_c=(beta_c["WAKE"][i, j] + beta_c["WAKE"][j, i]) / 2, rem_c=(beta_c["REM"][i, j] + beta_c["REM"][j, i]) / 2))
GL = pd.DataFrame(glm_rows)
print(f"GLMs fitted in {time.time() - t_glm:.0f} s over {GL.subject.nunique()} sessions")
print("pairs by type:", GL.groupby("pair").size().to_dict(), "; removed for a fit that did not converge or had ill-conditioned weights:",
      GL.groupby("pair").apply(lambda d: int((~np.isfinite(d.wake) | ~np.isfinite(d.rem)).sum())).to_dict())
print("paper: HD-HD 34,270, FS-FS 2,492, HD-FS 8,992 (Extended Data Fig. 9d gives 18,992)")
GLc = GL[np.isfinite(GL.wake) & np.isfinite(GL.rem)]
for p_ in ("HD-HD", "FS-FS", "HD-FS"):
    d = GLc[GLc.pair == p_]
    r, p = stats.pearsonr(d.wake, d.rem)
    print(f"{p_}: WAKE vs REM coupling, Pearson r = {r:.3f}, p = {p:.3g}, n = {len(d)} unordered pairs ({2 * len(d)} ordered)")
    dc = GL[(GL.pair == p_) & np.isfinite(GL.wake_c) & np.isfinite(GL.rem_c)]
    rs_ = stats.spearmanr(d.wake, d.rem)[0]
    print(f"   variant keeping ill-conditioned fits: r = {stats.pearsonr(dc.wake_c, dc.rem_c)[0]:.3f}, n = {len(dc)}; Spearman (main pairs) = {rs_:.3f}")
    print("   per session r, median and range:", d.groupby("subject").apply(lambda x: stats.pearsonr(x.wake, x.rem)[0] if len(x) > 2 else np.nan).describe()[["min", "50%", "max"]].round(3).to_dict())

**Result: reproduced (resting on an assumption).** In 27 sessions with at least 100 s of REM, WAKE and REM coupling coefficients are correlated for HD-HD pairs (r = 0.716, n = 25,361), FS-FS pairs (r = 0.400, n = 2,639) and HD-FS pairs (r = 0.436, n = 16,632), all p < 1e-100, with FS-FS below HD-HD. The HD-HD result depends on removing the 970 HD-HD pairs (3.7%) whose fits are ill-conditioned, standing for glmfit's warnings: with them kept the pooled HD-HD r is 0.075 (n = 26,311). Over the kept pairs the Spearman correlation is 0.795 and the per-session HD-HD r ranges from 0.41 to 0.85.

## ed9d-coupling-follows-tuning

> The polarity of the GLM coupling within a pair depends on the similarity of the two cells'
> HD tuning, during both WAKE and REM.

Pearson correlation across pairs between the zero-lag correlation of the two cells' square
tuning curves and the lag-0 coupling, for HD-FS and FS-FS pairs, in WAKE and in REM
(criterion; code Figure_GLMRemWake).

In [ ]:
for p_ in ("HD-FS", "FS-FS"):
    d = GLc[GLc.pair == p_]
    for st in ("wake", "rem"):
        r, p = stats.pearsonr(d.tc_r, d[st])
        print(f"{p_}, {st.upper()}: r = {r:.3f}, p = {p:.3g}, n = {len(d)}")
d = GLc[GLc.pair == "HD-HD"]
print("HD-HD, shown only:", [f"{st}: r = {stats.pearsonr(d.tc_r, d[st])[0]:.3f}" for st in ("wake", "rem")])

**Result: reproduced.** Coupling follows tuning similarity for HD-FS pairs (WAKE r = 0.760, REM r = 0.346, n = 16,632) and FS-FS pairs (WAKE r = 0.450, REM r = 0.144, n = 2,639), all p < 1e-13.

## fig7b-wake-isomap-tuning

> HD tuning curves of PoSub-HD and PoSub-FS cells computed from the virtual HD of the WAKE
> ring manifold are equivalent to those computed from the real HD.

Methods and GetIsomap_WAKE.m: HD cell spikes over the whole square epoch in 200 ms bins,
smoothed with 400 ms s.d., square root; Isomap with 12 neighbours and 3 dimensions; virtual
HD from the first two dimensions, its direction by the smaller circular variance of its
difference from real HD, its origin by the mean offset of maximum cross-correlation between
real and virtual curves over HD cells. Curves in 1 degree bins, 3 degree smoothing, from all
times of the epoch (code); real HD taken at each bin centre. Control: the virtual HD reversed
in time. Isomap is fitted on at most 4,000 vectors per session (helper; assumption, for time).

In [ ]:
ISO = {}
iso_rows = []
for si, s in enumerate(S):
    u = U[U.sess == si]
    a, b = s["epochs"]["wake_square"][0]
    edges = np.arange(a, b, 0.2); cen = edges[:-1] + 0.1
    t = s["hd_t"]
    j = np.clip(np.searchsorted(t, cen), 1, len(t) - 1); j = np.where(np.abs(t[j - 1] - cen) < np.abs(t[j] - cen), j - 1, j)
    real = np.where(np.abs(t[j] - cen) < 0.05, s["hd"][j], np.nan)
    hd_idx = np.where(u.hd.values)[0]
    X = posub.pop_vectors([s["spikes"][i] for i in hd_idx], edges)
    emb = posub.isomap_embed(X, 3, seed=700 + si)
    v = posub.virtual_hd(emb, real)
    allm = np.ones(len(cen), bool)
    h_real = posub.binned_tuning(s["spikes"], edges, real, allm)
    h_vir = posub.binned_tuning(s["spikes"], edges, v, allm)
    lag = posub.circ_xcorr_lag(h_real[:, hd_idx], h_vir[:, hd_idx])
    off = int(np.round(np.rad2deg(stats.circmean(np.deg2rad(lag))))) % 360
    v = v - np.deg2rad(off)  # the virtual curve rolled back by off bins to match the real one
    h_vir = posub.binned_tuning(s["spikes"], edges, v, allm)
    h_rev = posub.binned_tuning(s["spikes"], edges, v[::-1].copy(), allm)
    r = posub.zero_lag_r(h_real, h_vir); rc = posub.zero_lag_r(h_real, h_rev)
    rad = np.linalg.norm(emb[:, :2] - emb[:, :2].mean(0), axis=1)
    ISO[si] = dict(emb=emb, v=v, real=real, edges=edges, rad=rad)
    for i in np.where((u.hd | u.fs).values)[0]:
        iso_rows.append(dict(subject=s["subject"], cls="HD" if u.hd.values[i] else "FS", r=r[i], r_rev=rc[i]))
IR = pd.DataFrame(iso_rows)
for cls in ("HD", "FS"):
    d = IR[IR.cls == cls]
    show(signrank(d.r, d.r_rev), f"{cls}: real vs virtual curve correlation, against reversed control:")
print("paper: HD n = 1602, z = 34.7; FS n = 427, z = 17.9")
print("per session median r (HD, FS):", IR.groupby(["subject", "cls"]).r.median().unstack().round(2).T.to_string())
print(f"Isomap done at {time.time() - T0:.0f} s")

**Result: reproduced.** Tuning curves from the virtual HD of the WAKE manifold match those from the real HD for HD cells (median r 0.94 against -0.002 for the reversed control, |z| = 33.7, p = 1e-248, n = 1,512) and FS cells (0.84 against 0.02, |z| = 17.95, p = 4e-72, n = 430).

## ed10b-ring-topology

> Population vectors of PoSub-HD cells lie on a ring during WAKE, farther from the centre of
> the manifold than shuffled vectors, and the ring has the same radius during REM.

Distance of each point to the centre of the first two Isomap dimensions, normalized to the
mean distance of the real projection. Shuffle: each cell's binned train shifted circularly by
a random number of bins, then embedded the same way. Real against shuffled mean distance,
Wilcoxon signed-rank across sessions. REM against WAKE: the joint embedding below (WAKE
vectors subsampled to the number of REM vectors), mean distance per session in each state,
Wilcoxon signed-rank (a claim of absence, judged at p < 0.05).

In [ ]:
rng = np.random.default_rng(10)
ring = []
for si, s in enumerate(S):
    u = U[U.sess == si]; hd_idx = np.where(u.hd.values)[0]
    e = ISO[si]["edges"]
    C = posub.bin_counts([s["spikes"][i] for i in hd_idx], e)
    C = np.stack([np.roll(C[:, k], rng.integers(len(C))) for k in range(C.shape[1])], 1)
    Xs = np.sqrt(gaussian_filter1d(C, 2, axis=0))
    es = posub.isomap_embed(Xs, 3, seed=900 + si)
    rr = ISO[si]["rad"]; rs = np.linalg.norm(es[:, :2] - es[:, :2].mean(0), axis=1)
    ring.append(dict(subject=s["subject"], real=1.0, shuffled=rs.mean() / rr.mean(), real_cv=rr.std() / rr.mean(), shuffled_cv=rs.std() / rs.mean()))
ring = pd.DataFrame(ring)
print(ring.round(3).to_string())
show(signrank(ring.real, ring.shuffled), "mean normalized distance, real (1 by construction) vs shuffled:")
print("paper: n = 32, z = 4.79")

**Result: reproduced.** The real population vectors lie farther from the manifold centre than shuffled ones in every session (shuffled mean distance 0.33 to 0.65 of the real, signed-rank p = 1.2e-6, n = 31; paper z = 4.79), and in the joint WAKE-REM embedding the mean REM distance does not differ from WAKE (median ratio 0.96, p = 0.43, n = 27; paper z = 0.17).

## fig7d-rem-isomap-tuning

> During REM sleep the HD tuning of PoSub-HD and PoSub-FS cells can be estimated from the
> virtual HD of the ring manifold alone and matches their WAKE tuning.

Sessions with at least 100 s of REM (Methods). HD-cell population vectors (200 ms, 400 ms
s.d., square root) in REM and in the square epoch; WAKE vectors subsampled at random to the
number of REM vectors; Isomap (12 neighbours, 3 dimensions) on both together; virtual HD set
in direction and origin from the WAKE vectors' real HD. Curves in 6 degree bins with 6 degree
smoothing. Cells with REM rate above 0.5 Hz (HD) or 10 Hz (FS). WAKE curve read (i) as the
WAKE Isomap curve and (ii) as the real-HD WAKE curve; control, the REM curve from the REM
virtual HD reversed in time.

In [ ]:
rng = np.random.default_rng(7)
rem_rows, rad_rows = [], []
for si, s in enumerate(S):
    remiv = s["sleep"].get("rem", np.zeros((0, 2)))
    if (remiv[:, 1] - remiv[:, 0]).sum() < 100:
        continue
    u = U[U.sess == si]; hd_idx = np.where(u.hd.values)[0]
    tmax = max(b for v in s["epochs"].values() for a, b in v)
    edges = np.arange(0, tmax, 0.2); cen = edges[:-1] + 0.1
    in_rem = posub.in_intervals(cen, remiv)
    a, b = s["epochs"]["wake_square"][0]
    in_wk = (cen >= a) & (cen < b)
    wk_idx = np.sort(rng.choice(np.where(in_wk)[0], in_rem.sum(), replace=False))
    rem_idx = np.where(in_rem)[0]
    X = posub.pop_vectors([s["spikes"][i] for i in hd_idx], edges)
    sel = np.r_[wk_idx, rem_idx]
    emb = posub.isomap_embed(X[sel], 3, seed=1100 + si)
    t = s["hd_t"]
    j = np.clip(np.searchsorted(t, cen[sel]), 1, len(t) - 1); j = np.where(np.abs(t[j - 1] - cen[sel]) < np.abs(t[j] - cen[sel]), j - 1, j)
    real = np.where(np.abs(t[j] - cen[sel]) < 0.05, s["hd"][j], np.nan)
    nw = len(wk_idx); real[nw:] = np.nan
    v = posub.virtual_hd(emb, real)  # direction from the WAKE vectors (REM real HD is NaN)
    ang_full = np.full(len(cen), np.nan); ang_full[sel] = v
    real_full = np.full(len(cen), np.nan); real_full[sel[:nw]] = real[:nw]
    mw = np.zeros(len(cen), bool); mw[wk_idx] = True
    mr = np.zeros(len(cen), bool); mr[rem_idx] = True
    kw = dict(nbins=60, sd=1)
    hw_real = posub.binned_tuning(s["spikes"], edges, real_full, mw, **kw)
    hw_vir = posub.binned_tuning(s["spikes"], edges, ang_full, mw, **kw)
    lag = posub.circ_xcorr_lag(hw_real[:, hd_idx], hw_vir[:, hd_idx])
    off = stats.circmean(np.deg2rad(lag * 6))
    ang_full = ang_full - off
    hw_vir = posub.binned_tuning(s["spikes"], edges, ang_full, mw, **kw)
    hr_vir = posub.binned_tuning(s["spikes"], edges, ang_full, mr, **kw)
    rev = ang_full.copy(); rev[rem_idx] = ang_full[rem_idx][::-1]
    hr_rev = posub.binned_tuning(s["spikes"], edges, rev, mr, **kw)
    rem_dur = (remiv[:, 1] - remiv[:, 0]).sum()
    rate_rem = np.array([posub.in_intervals(sp, remiv).sum() / rem_dur for sp in s["spikes"]])
    ri = posub.zero_lag_r(hw_vir, hr_vir); rii = posub.zero_lag_r(hw_real, hr_vir)
    ci = posub.zero_lag_r(hw_vir, hr_rev); cii = posub.zero_lag_r(hw_real, hr_rev)
    for i in range(len(u)):
        cls = "HD" if u.hd.values[i] and rate_rem[i] > 0.5 else ("FS" if u.fs.values[i] and rate_rem[i] > 10 else None)
        if cls:
            rem_rows.append(dict(subject=s["subject"], cls=cls, r_iso=ri[i], r_real=rii[i], c_iso=ci[i], c_real=cii[i]))
    rad = np.linalg.norm(emb[:, :2] - emb[:, :2].mean(0), axis=1); rad = rad / rad[:nw].mean()
    wv = posub.zero_lag_r(hw_real[:, hd_idx], hw_vir[:, hd_idx])
    rad_rows.append(dict(subject=s["subject"], rem_s=round(rem_dur), n_vectors=int(in_rem.sum()), wake=rad[:nw].mean(), rem=rad[nw:].mean(),
                         wake_real_vs_virtual_r=np.nanmedian(wv)))
RR = pd.DataFrame(rem_rows); RD = pd.DataFrame(rad_rows)
print(RD.round(3).to_string())
print("sessions with at least 100 s of REM:", len(RD), "(paper 26 of 32)")
for cls in ("HD", "FS"):
    d = RR[RR.cls == cls]
    show(signrank(d.r_iso, d.c_iso), f"{cls}, reading (i) WAKE Isomap curve vs REM, against control:")
    show(signrank(d.r_real, d.c_real), f"{cls}, reading (ii) WAKE real-HD curve vs REM, against control:")
print("paper: HD n = 1148, z = 29.2; FS n = 317, z = 10.4")
print("ed10b, REM vs WAKE radius in the joint embedding:")
show(signrank(RD.rem, RD.wake), "   mean normalized distance, REM vs WAKE:")
print("paper: n = 26, z = 0.17")

**Result: reproduced.** In 27 sessions with at least 100 s of REM, REM tuning curves from the virtual HD match WAKE curves above the time-reversed control for HD cells (median r 0.86, |z| = 30.3, n = 1,251) and FS cells (0.42, |z| = 11.5, n = 332), with the WAKE curve from the Isomap angle, and the same with the real-HD WAKE curve (0.82 and 0.38, |z| = 30.3 and 11.7); all p < 1e-30.

## fig1ac-posub-border (figshare, data elsewhere)

> Along a Neuropixel probe crossing pRSC and PoSub, HD tuning of excitatory cells rises in a
> step at the functional border between the two regions.

The Neuropixel session is figshare Dataset_4.tar.gz (claims file), streamed here. Excitatory
cells as for PoSub (rate at least 0.5 Hz in the square epoch, trough-to-peak above 0.4 ms
from the file's WaveformFeatures, rate below 10 Hz); HD information from the square-epoch
curve at speed above 2 cm/s (file's Velocity). Depth from CellDepth.mat (micrometres), set
to 0 at the shallowest cell as the code sets its depth. Running average as
Figure_SingleRec_NPX.m: mean HD information in 50 um bins from 0 to -2.5 mm, empty bins
interpolated, Gaussian smoothing of 2 bins. The judgement is qualitative (criterion).

In [ ]:
NP = posub.load_adn(file_id=posub.FIGSHARE_NPX_FILE)[0]
tunN = posub.adn_tuning(NP)
ttpN = NP["mat"]["tr2pk"].ravel(); depN = NP["mat"]["cellDep"].ravel().astype(float)
depN = (depN - depN.max()) / 1000
exN = (tunN["rate"] >= 0.5) & np.isfinite(ttpN) & (ttpN > 0.4) & (tunN["rate"] < 10)
print("Neuropixel session", NP["session"], "; units", len(NP["spikes"]), "; excitatory", int(exN.sum()),
      "; depth range (mm)", depN.min(), depN.max(), "; the files' own excitatory count", int(NP["file_types"]["ex"].sum()))
top = -0.05 * np.arange(50); br = np.c_[top, top - 0.05]
prof = np.array([tunN["info"][exN & (depN < a) & (depN >= b)].mean() if (exN & (depN < a) & (depN >= b)).any() else np.nan for a, b in br])
ok = np.isfinite(prof); tt = np.arange(len(prof))
prof_i = prof.copy(); prof_i[~ok] = np.interp(tt[~ok], tt[ok], prof[ok])
prof_s = gaussian_filter1d(prof_i, 2)
mid = br.mean(1)
for d_, v_ in zip(mid[::4], prof_s[::4]):
    print(f"depth {d_:6.3f} mm: running mean HD information {v_:.3f} bits/spike")
dpr = np.diff(prof_s)
k = int(np.argmax(dpr if prof_s[-1] > prof_s[0] else -dpr))
print(f"largest change between adjacent bins at {mid[k]:.3f} mm ({dpr[k]:+.3f}); profile range {prof_s.min():.3f} to {prof_s.max():.3f}; "
      f"shallow third mean {prof_s[: len(prof_s) // 3].mean():.3f}, deep third mean {prof_s[-len(prof_s) // 3:].mean():.3f}")
half = (prof_s.max() + prof_s.min()) / 2
cross = np.where(np.diff(np.sign(prof_s - half)) != 0)[0]
print("crossings of the half-range level at depths (mm):", mid[cross].round(3))
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].scatter(tunN["info"][exN], depN[exN], s=6); ax[0].set_xlabel("HD info (bits/spike)"); ax[0].set_ylabel("depth (mm)")
ax[1].plot(prof_s, mid); ax[1].set_xlabel("running mean HD info"); plt.tight_layout(); plt.show()
print("plotted ranges: info", tunN["info"][exN].min().round(3), tunN["info"][exN].max().round(3))

**Result: reproduced (qualitative).** Along the figshare Neuropixel probe the running mean HD information of 134 excitatory cells stays near 0.01 to 0.24 bits per spike over the shallow 1.4 mm and lies between 0.78 and 1.19 from 1.8 mm down, crossing its half-range level once, at -1.675 mm (shallow third mean 0.074, deep third 0.940); the profile has a smaller bump near -0.6 mm (0.19).

## Claims not tested

- fig4-random-connectivity-simulation. A result of simulation. The paper's Python
  simulation code is not in the figshare item, and the Methods and Supplementary Methods do
  not give the input tuning widths, the number of inputs or the log-normal weight parameters
  of each point.
- fig4h-model-fit. The PyTorch model fit is not deposited, and the Supplementary Methods
  do not give the numbers of inputs or the initial values.
- ed6-trn-innervates-adn. Anatomical tracing images, which are neither in the dandiset nor
  in the figshare item.

## Summary

| Claim | Data | Outcome |
|---|---|---|
| cell-classes-counts | dandiset | partially reproduced (parallel unit count short by one session) |
| ed2b-excitatory-hd-information | dandiset | reproduced |
| fig2a-fs-hd-information | dandiset | reproduced |
| fig2b-decoding-from-fs | dandiset | reproduced |
| fig2c-geometry-independent | dandiset | reproduced |
| ed2hi-within-epoch-stability | dandiset | reproduced |
| fig2f-cue-rotation | figshare Dataset_3 | reproduced |
| fig3c-adn-broader | figshare Dataset_5 | reproduced |
| fig3-ten-components-power | dandiset | partially reproduced (holds for squared-modulus power only) |
| fig3e-fourier-signatures-differ | figshare Dataset_5 | not reproduced (assumed: squared-modulus power, numeric cell type) |
| fig3f-posub-hd-closer-to-fs | figshare Dataset_5 | partially reproduced (p < 1e-9 under one of four readings) |
| ed3k-robust-to-controls | figshare Dataset_5 | partially reproduced (multipeak scores lower in ADN, a question is asked) |
| fig3g-first-three-components | dandiset | reproduced (assumed: squared-modulus power) |
| fig3h-dominant-component | dandiset | partially reproduced |
| ed3g-fs-spectra-heterogeneous | dandiset | reproduced |
| ed3l-fs-phases-independent | dandiset | reproduced |
| ed4a-autocorrelogram-symmetries | dandiset | reproduced |
| ed4cd-isomap-posub-central | figshare Dataset_5 | reproduced |
| fig4-random-connectivity-simulation | none deposited | blocked |
| fig4f-synaptic-weights-dispersed | dandiset | reproduced |
| fig4h-model-fit | none deposited | blocked |
| ed7-adn-disinhibition | figshare Dataset_6 | reproduced |
| fig5b-posub-rates-increase | dandiset | reproduced |
| ed7gj-tuning-preserved | dandiset | reproduced |
| fig5e-hd-multiplicative-fs-additive | dandiset | partially reproduced (one control comparison fails under one exclusion reading) |
| ed8cd-multiplicative-detectable | dandiset | reproduced (assumed: titrated noise) |
| ed8e-adn-mixed-gain | figshare Dataset_6 | reproduced |
| fig6c-coupling-preserved-wake-rem | dandiset | reproduced (assumed: removal of ill-conditioned fits) |
| ed9d-coupling-follows-tuning | dandiset | reproduced |
| fig7b-wake-isomap-tuning | dandiset | reproduced |
| ed10b-ring-topology | dandiset | reproduced |
| fig7d-rem-isomap-tuning | dandiset | reproduced |
| fig1ac-posub-border | figshare Dataset_4 | reproduced (qualitative) |
| ed6-trn-innervates-adn | none deposited | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. Python reimplementations of the MATLAB code
(tuning curves, HD information, Bayesian decoding, Poisson GLMs by Newton-Raphson, CCG
synapse detection); Isomap from scikit-learn, fitted on at most 4,000 population vectors per
session; speed computed from the position, as the files hold no speed series; only lag 0 of
the GLM; the dandiset's 31 sessions in place of the paper's 32, and 2 of the 5 ArchT mice for
the PoSub optogenetic claims; cue rotation realignment from a Methods-based reimplementation of
the code's sigmoid procedure.

What was found about the dataset. The dandiset holds 31 of the paper's 32 PoSub sessions:
all 14 vertical ones (931 units, as the paper gives) and 17 of 18 parallel ones (1,760 units
against 1,999). Of the five ArchT mice of the PoSub optogenetic experiment it holds two
(A1813, A1815); the other three (A1801, A1804, A1806, six sessions) are on figshare only, as
are the ADN, cue rotation and Neuropixel recordings. The optogenetic stimulus series of eight
files holds four interleaved channels in one dimension, so its length at the stated 20 kHz is
four times the session's; in A1808 it holds one channel of 0 and 1. The dandiset's unit
labels (is_excitatory, is_fast_spiking, is_head_direction) follow the code's 0.35 ms cut, not
the Methods' 0.4 ms. The files hold no speed series and the probe orientation is given only
in the device description.